In [ ]:


# -*- coding: utf-8 -*-
"""
================================================================================
BRISC2025 — Kaggle : Joint_Total + SwinHAFNet + Inférence Corrigée
================================================================================
CORRECTION DES ERREURS :
  1. UNetClassifier : Down(512,512) + Linear(512,256) + Linear(256,4)
     → correspond EXACTEMENT aux checkpoints .pth dans result_model
  2. DoubleConv     : self.double_conv (pas self.net)
  3. Down           : self.maxpool_conv (pas self.mp_conv)
  4. LR_JOINT       : 5e-4 (pas 1e-3 — trop élevé pour convergence)
  5. FileNotFoundError : out_dir.mkdir(parents=True, exist_ok=True) systématique

SOURCE DES DONNÉES :
  result_model/processed_data/  → données prétraitées .npy
  result_model/models/          → .pth (clf, att, joint, seg)

SORTIES :
  /kaggle/working/resultat_joint.zip → processed_données + models + results + inference
  /kaggle/working/resultat_swin.zip  → model swin + results

MÉTRIQUES AFFICHÉES (identiques code original) :
  Train/Val/Test : Loss, IoU, Dice, Accuracy, F1
  Inference      : Dice par image + Dice agrégé (comparable entraînement)
  Visualisations : 9 prédictions annotées + superposition toutes images + courbes
================================================================================
"""

# ============================================================================
# BLOC 1 : IMPORTS ET CONFIGURATION
# ============================================================================
import os, sys, json, shutil, time, zipfile, warnings, math
from pathlib import Path
from collections import defaultdict
from datetime import datetime
import contextlib

import numpy as np
import pandas as pd
import cv2
from PIL import Image

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.cuda.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    confusion_matrix, classification_report,
    f1_score, balanced_accuracy_score,
    precision_score, recall_score,
    precision_recall_fscore_support,
)

# Installation timm si nécessaire
try:
    import timm
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm'])
    import timm

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
print(f"✅ PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}")

# ── GPU ───────────────────────────────────────────────────────────────────────
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark     = True
    torch.backends.cudnn.deterministic = False
    print(f"✅ GPU : {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

def _amp():
    """Context manager AMP (autocast si GPU, nullcontext sinon)."""
    return autocast() if USE_AMP else contextlib.nullcontext()

# ── Constantes ────────────────────────────────────────────────────────────────
CLASS_NAMES  = ['glioma', 'meningioma', 'no_tumor', 'pituitary']
N_CLASSES    = 4
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {i: c for i, c in enumerate(CLASS_NAMES)}
CLF_LABELS   = list(range(N_CLASSES))
SEG_THRESH   = 0.5

# ── Hyperparamètres (IDENTIQUES au code Kaggle original) ─────────────────────
HYPERPARAMS = {
    'LR_CLF'             : 1e-4,
    'LR_SEG'             : 1e-3,
    'LR_JOINT'           : 5e-4,   # ← CORRECTION (était 1e-3, trop élevé)
    'LR_SWIN'            : 5e-5,   # faible car backbone pré-entraîné
    'BATCH_CLF'          : 32,
    'BATCH_SEG'          : 16,
    'BATCH_JOINT'        : 12,
    'BATCH_SWIN'         : 8,
    'EPOCHS'             : 100,
    'EARLY_STOP_PATIENCE': 15,
    'WEIGHT_DECAY'       : 1e-3,
    'POS_WEIGHT'         : 10.0,
    'SEG_LOSS_W'         : 0.7,
    'CLF_LOSS_W'         : 0.3,
}

# ── Chemins Kaggle ────────────────────────────────────────────────────────────
KAGGLE_INPUT   = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')

# Détection result_model (ajouté comme dataset Kaggle)
def find_result_model():
    """Cherche result_model dans /kaggle/input/."""
    for root, dirs, _ in os.walk(KAGGLE_INPUT):
        for d in dirs:
            p = Path(root) / d
            if (p / 'processed_data').exists() and (p / 'models').exists():
                return p
    # Fallback : chercher models/ avec .pth
    for p in KAGGLE_INPUT.rglob('unet_clf_best.pth'):
        return p.parent.parent
    raise FileNotFoundError(
        "result_model introuvable dans /kaggle/input/\n"
        "Ajoutez le dataset result_model via 'Add Data' dans Kaggle"
    )

RESULT_MODEL_DIR = find_result_model()
INPUT_PROC_DIR   = RESULT_MODEL_DIR / 'processed_data'
INPUT_MODELS_DIR = RESULT_MODEL_DIR / 'models'

print(f"\n✅ result_model trouvé : {RESULT_MODEL_DIR}")
print(f"   processed_data : {INPUT_PROC_DIR}")
print(f"   models         : {INPUT_MODELS_DIR}")

# Vérification des .pth disponibles
for pth_name in ['unet_clf_best.pth', 'att_unet_best.pth',
                  'unet_joint_best.pth', 'unet_seg_best.pth']:
    p = INPUT_MODELS_DIR / pth_name
    status = f"✅ {p.stat().st_size/1e6:.1f} Mo" if p.exists() else "❌ ABSENT"
    print(f"   {pth_name:25s} : {status}")

# ── Dossiers de travail (écriture dans /kaggle/working) ──────────────────────
WORK_DIR    = KAGGLE_WORKING / 'brisc_work'
PROC_DIR    = WORK_DIR / 'processed_donnees'   # ← nom imposé
MODELS_DIR  = WORK_DIR / 'models'
RESULTS_DIR = WORK_DIR / 'results'
PLOTS_DIR   = RESULTS_DIR / 'plots'
TABLES_DIR  = RESULTS_DIR / 'tables'
INF_DIR     = RESULTS_DIR / 'inference'

for _d in [WORK_DIR, PROC_DIR, MODELS_DIR, RESULTS_DIR, PLOTS_DIR, TABLES_DIR, INF_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# Copier les modèles .pth dans le dossier de travail
print("\n📦 Copie des modèles .pth vers WORK_DIR/models ...")
for pth_name in ['unet_clf_best.pth', 'att_unet_best.pth',
                  'unet_joint_best.pth', 'unet_seg_best.pth']:
    src = INPUT_MODELS_DIR / pth_name
    dst = MODELS_DIR / pth_name
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)
        print(f"  Copié : {pth_name}")

# Stockage global
all_histories   = {}
results_all     = {}
complexity_info = {}

print("✅ Bloc 1 terminé")

# ============================================================================
# BLOC 2 : STRUCTURE DES DONNÉES PRÉTRAITÉES
# ============================================================================
"""
On utilise les données déjà prétraitées depuis result_model/processed_data/
Structure attendue :
  processed_data/
    class_mapping.json
    train/{glioma, meningioma, no_tumor, pituitary}/*.npy  (5000 images)
    test/ {glioma, meningioma, no_tumor, pituitary}/*.npy  (1000 images)
    segmentation/train/images/*.npy   (3933)
    segmentation/train/masks/*.npy    (3933)
    segmentation/test/images/*.npy    (860)
    segmentation/test/masks/*.npy     (860)

On CRÉE en plus :
  processed_donnees/segmentation_new/  (5000 train + 1000 test avec no_tumor)
"""
print("\n" + "="*70)
print("📂 BLOC 2 : VÉRIFICATION ET COPIE DES DONNÉES PRÉTRAITÉES")
print("="*70)

# Copier processed_data → processed_donnees (nom imposé par le projet)
_copy_flag = PROC_DIR / '.data_ready'
if not _copy_flag.exists():
    print("  Copie processed_data → processed_donnees ...")
    # Copier class_mapping.json
    _src_map = INPUT_PROC_DIR / 'class_mapping.json'
    if _src_map.exists():
        shutil.copy2(_src_map, PROC_DIR / 'class_mapping.json')
    else:
        # Créer un mapping par défaut
        (PROC_DIR / 'class_mapping.json').write_text(
            json.dumps(CLASS_TO_IDX, indent=2))

    # Copier tous les sous-dossiers
    for sub in INPUT_PROC_DIR.iterdir():
        if sub.is_dir():
            dst = PROC_DIR / sub.name
            if not dst.exists():
                shutil.copytree(str(sub), str(dst))
                n_npy = len(list(dst.rglob('*.npy')))
                print(f"  Copié : {sub.name}/ ({n_npy} .npy)")
    _copy_flag.touch()
    print("  ✅ Copie terminée")
else:
    print("  ✅ Données déjà copiées (skip)")

# Charger le mapping
_map_path = PROC_DIR / 'class_mapping.json'
if _map_path.exists():
    with open(_map_path) as f:
        _cls_map = json.load(f)
else:
    _cls_map = CLASS_TO_IDX

# Vérification comptages
print("\n📊 VÉRIFICATION DES DONNÉES :")
_checks = [
    ('classification/train', PROC_DIR / 'train',                               4900, 5100),
    ('classification/test',  PROC_DIR / 'test',                                 990, 1010),
    ('seg/train/images',     PROC_DIR / 'segmentation' / 'train' / 'images',   3900, 3950),
    ('seg/test/images',      PROC_DIR / 'segmentation' / 'test'  / 'images',    850,  870),
]
for name, path, lo, hi in _checks:
    n = len(list(path.rglob('*.npy'))) if path.exists() else 0
    ok = lo <= n <= hi
    print(f"  {'✅' if ok else '❌'} {name:30s} : {n:5d}  (attendu [{lo}-{hi}])")

print("✅ Bloc 2 terminé")

# ============================================================================
# BLOC 3 : CRÉATION DE segmentation_new (5000 train + 1000 test AVEC no_tumor)
# ============================================================================
"""
segmentation_new = segmentation standard + images no_tumor avec MASQUES NOIRS

Train (5000) : 3933 seg réelles + 1067 no_tumor (masques zeros)
Test  (1000) : 860  seg réelles + 140  no_tumor (masques zeros)

MASQUES NOIRS :
  - np.zeros((224,224), float32) prétraité identiquement aux vrais masques
  - Même nom de fichier que l'image .npy correspondante
  - EXCLUS du calcul Dice/IoU (GT tout-zéro = pas de tumeur à évaluer)
  - Le modèle apprend : si no_tumor → prédire masque quasi-vide (correct)
"""
print("\n" + "="*70)
print("📂 BLOC 3 : CRÉATION segmentation_new/ (avec no_tumor + masques noirs)")
print("="*70)

SEG_NEW_DIR = PROC_DIR / 'segmentation_new'
_sn_flag    = SEG_NEW_DIR / '.created'

if not _sn_flag.exists():
    seg_proc = PROC_DIR / 'segmentation'  # déjà prétraité

    for split, expected_total in [('train', 5000), ('test', 1000)]:
        out_img = SEG_NEW_DIR / split / 'images'
        out_msk = SEG_NEW_DIR / split / 'masks'
        out_img.mkdir(parents=True, exist_ok=True)
        out_msk.mkdir(parents=True, exist_ok=True)

        n_seg = n_no_tumor = 0

        # 1) Copier les images de segmentation existantes (déjà prétraitées)
        seg_img_src = seg_proc / split / 'images'
        seg_msk_src = seg_proc / split / 'masks'
        if seg_img_src.exists():
            for img_p in sorted(seg_img_src.glob('*.npy')):
                msk_p = seg_msk_src / img_p.name
                if msk_p.exists():
                    shutil.copy2(img_p, out_img / img_p.name)
                    shutil.copy2(msk_p, out_msk / img_p.name)
                    n_seg += 1

        # 2) Copier les images no_tumor + créer masques noirs
        #    Les images no_tumor sont dans classification/{split}/no_tumor/
        nt_clf_dir = PROC_DIR / split / 'no_tumor'
        if not nt_clf_dir.exists():
            # Chercher aussi dans processed_data/test/no_tumor
            nt_clf_dir = INPUT_PROC_DIR / split / 'no_tumor'

        black_mask = np.zeros((224, 224), dtype=np.float32)

        if nt_clf_dir.exists():
            for img_p in sorted(nt_clf_dir.glob('*.npy')):
                shutil.copy2(img_p, out_img / img_p.name)
                # Masque noir avec MÊME NOM que l'image
                np.save(out_msk / img_p.name, black_mask)
                n_no_tumor += 1
        else:
            print(f"  ⚠️  {nt_clf_dir} absent → no_tumor ignoré pour {split}")

        total = n_seg + n_no_tumor
        print(f"  [{split.upper()}] seg:{n_seg} + no_tumor:{n_no_tumor} "
              f"= {total} (attendu: {expected_total})")

    _sn_flag.touch()
    print("  ✅ segmentation_new créée")
else:
    # Vérification rapide
    for split in ['train', 'test']:
        n_img = len(list((SEG_NEW_DIR/split/'images').glob('*.npy'))) if \
                (SEG_NEW_DIR/split/'images').exists() else 0
        n_msk = len(list((SEG_NEW_DIR/split/'masks').glob('*.npy'))) if \
                (SEG_NEW_DIR/split/'masks').exists() else 0
        print(f"  ✅ seg_new/{split}: {n_img} images | {n_msk} masques")

# Vérification cohérence images ↔ masques
for split in ['train', 'test']:
    _i = set(p.stem for p in (SEG_NEW_DIR/split/'images').glob('*.npy'))
    _m = set(p.stem for p in (SEG_NEW_DIR/split/'masks').glob('*.npy'))
    _diff = _i.symmetric_difference(_m)
    if _diff:
        print(f"  ⚠️  {split}: {len(_diff)} fichiers sans correspondance")
    else:
        print(f"  ✅ {split}: {len(_i)} paires image+masque cohérentes")

print("✅ Bloc 3 terminé")

# ============================================================================
# BLOC 4 : ARCHITECTURES (IDENTIQUES AU CODE KAGGLE ORIGINAL)
# ============================================================================
"""
CORRECTION ARCHITECTURE — CAUSE DES ERREURS :
  UNetClassifier pth : down4=Down(512,512), classifier=Linear(512,256)+Linear(256,4)
  UNetClassifier code précédent : down4=Down(512,1024) → MAUVAIS
  → Correction : utiliser EXACTEMENT la même architecture que le code Kaggle original

  DoubleConv : self.double_conv (identique code original)
  Down       : self.maxpool_conv (identique code original)
  Up         : self.up + self.conv (identique code original)
"""
print("\n" + "="*70)
print("🧠 BLOC 4 : ARCHITECTURES (IDENTIQUES AU CODE ORIGINAL)")
print("="*70)

# ── Blocs de base (nommage IDENTIQUE au code Kaggle original) ─────────────────
class DoubleConv(nn.Module):
    """
    Bloc de convolution double : Conv3x3→BN→ReLU→Conv3x3→BN→ReLU
    IDENTIQUE au code original : self.double_conv (pas self.net ou self.block)
    bias=False car BatchNorm rend le biais redondant.
    """
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        # Nom IDENTIQUE au code Kaggle : self.double_conv
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """
    Downsampling : MaxPool2d(2) → DoubleConv
    IDENTIQUE au code original : self.maxpool_conv (pas self.mp_conv)
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # Nom IDENTIQUE au code Kaggle : self.maxpool_conv
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )
    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    """
    Upsampling : ConvTranspose2d ou bilinear → concat skip → DoubleConv
    IDENTIQUE au code original.
    """
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up   = nn.ConvTranspose2d(in_channels, in_channels // 2,
                                            kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)
    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX//2, diffX-diffX//2, diffY//2, diffY-diffY//2])
        return self.conv(torch.cat([x2, x1], dim=1))


class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
    def forward(self, x):
        return self.conv(x)


# ── UNetClassifier (9.54M) ── ARCHITECTURE CORRIGÉE ──────────────────────────
class UNetClassifier(nn.Module):
    """
    UNet adapté pour la classification.
    ✅ CORRECTION : down4=Down(512,512) + Linear(512,256) + Linear(256,n_classes)
    C'est l'architecture EXACTE des checkpoints .pth fournis.
    Le code précédent utilisait Down(512,1024) → size mismatch.
    """
    def __init__(self, n_channels=1, n_classes=4):
        super().__init__()
        self.inc   = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 512)          # ← CORRECTION : 512 pas 1024
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(512, 256),             # ← CORRECTION : 512→256 pas 1024→512
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, n_classes)        # ← CORRECTION : 256→n_classes
        )
    def forward(self, x):
        x = self.inc(x)
        x = self.down1(x); x = self.down2(x)
        x = self.down3(x); x = self.down4(x)
        return self.classifier(x)


# ── UNetWithClassifier / Joint (31.56M) ──────────────────────────────────────
class UNetWithClassifier(nn.Module):
    """
    UNet avec double sortie : segmentation + classification.
    IDENTIQUE au code Kaggle original :
      down4=Down(512,1024), classifier=Linear(1024,512)+Linear(512,n_clf_classes)
    Pour joint_total : entraîné sur 5000 images (avec no_tumor).
    Sauvegardé sous : joint_total.pth
    """
    def __init__(self, n_channels=1, n_seg_classes=1, n_clf_classes=4, bilinear=False):
        super().__init__()
        factor = 2 if bilinear else 1
        # Encodeur partagé
        self.inc   = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024 // factor)
        # Décodeur segmentation
        self.up1   = Up(1024, 512 // factor, bilinear)
        self.up2   = Up(512,  256 // factor, bilinear)
        self.up3   = Up(256,  128 // factor, bilinear)
        self.up4   = Up(128,  64,            bilinear)
        self.outc  = OutConv(64, n_seg_classes)
        # Tête classification
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(1024 // factor, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, n_clf_classes)
        )
    def forward(self, x):
        x1 = self.inc(x); x2 = self.down1(x1)
        x3 = self.down2(x2); x4 = self.down3(x3); x5 = self.down4(x4)
        clf_logits = self.classifier(x5)
        x = self.up1(x5, x4); x = self.up2(x, x3)
        x = self.up3(x, x2);  x = self.up4(x, x1)
        return self.outc(x), clf_logits


# ── AttentionGate + AttentionUp + AttentionUNet (31.39M) ─────────────────────
class AttentionGate(nn.Module):
    """
    Gate d'attention : pondère les skip connections.
    IDENTIQUE au code Kaggle original.
    """
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g  = nn.Sequential(nn.Conv2d(F_g, F_int, kernel_size=1),
                                   nn.BatchNorm2d(F_int))
        self.W_x  = nn.Sequential(nn.Conv2d(F_l, F_int, kernel_size=1),
                                   nn.BatchNorm2d(F_int))
        self.psi  = nn.Sequential(nn.Conv2d(F_int, 1, kernel_size=1),
                                   nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        g1 = self.W_g(g); x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        return x * self.psi(psi)


class AttentionUp(nn.Module):
    """IDENTIQUE au code Kaggle original."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_channels, in_channels // 2,
                                        kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)
        self.att  = AttentionGate(in_channels // 2, in_channels // 2,
                                   out_channels // 2)
    def forward(self, x1, x2):
        x1 = self.up(x1); x2 = self.att(g=x1, x=x2)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX//2, diffX-diffX//2, diffY//2, diffY-diffY//2])
        return self.conv(torch.cat([x2, x1], dim=1))


class AttentionUNet(nn.Module):
    """
    UNet avec AttentionGates — 31.39M paramètres.
    IDENTIQUE au code Kaggle original.
    """
    def __init__(self, n_channels=1, n_classes=1):
        super().__init__()
        self.inc   = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128); self.down2 = Down(128, 256)
        self.down3 = Down(256, 512); self.down4 = Down(512, 1024)
        self.up1   = AttentionUp(1024, 512); self.up2 = AttentionUp(512, 256)
        self.up3   = AttentionUp(256, 128);  self.up4 = AttentionUp(128, 64)
        self.outc  = OutConv(64, n_classes)
    def forward(self, x):
        x1=self.inc(x); x2=self.down1(x1); x3=self.down2(x2)
        x4=self.down3(x3); x5=self.down4(x4)
        x=self.up1(x5,x4); x=self.up2(x,x3); x=self.up3(x,x2); x=self.up4(x,x1)
        return self.outc(x)


# ── SwinHAFNet (~28M) ─────────────────────────────────────────────────────────
class SwinHAFNet(nn.Module):
    """
    Swin Transformer Hybrid Attention Fusion Network.
    Entrée  : (B, 1, 224, 224)
    Sorties : seg_logits(B,1,224,224) + clf_logits(B,4)

    Backbone : Swin-Tiny pré-entraîné ImageNet (features_only=True)
    Décodeur : style UNet avec skip connections
    clf_head : depuis le bottleneck (stage 3, 768 canaux)

    Référence : Swin-Unet https://arxiv.org/abs/2105.05537
    Code      : https://github.com/HuCaoFighting/Swin-Unet
    """
    def __init__(self, n_clf_classes=4, pretrained=True):
        super().__init__()
        # Adaptateur grayscale → 3 canaux (Swin attend RGB)
        self.chan_adapt = nn.Sequential(
            nn.Conv2d(1, 3, 1, bias=False),
            nn.BatchNorm2d(3)
        )
        # Backbone Swin-Tiny
        self.backbone = timm.create_model(
            'swin_tiny_patch4_window7_224',
            pretrained=pretrained,
            features_only=True,
            out_indices=(0, 1, 2, 3)
        )
        # Décodeur (4 blocs conv avec skip connections)
        self.dec3 = nn.Sequential(
            nn.Conv2d(768+384, 384, 3, padding=1, bias=False),
            nn.BatchNorm2d(384), nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )
        self.dec2 = nn.Sequential(
            nn.Conv2d(256+192, 192, 3, padding=1, bias=False),
            nn.BatchNorm2d(192), nn.ReLU(inplace=True),
            nn.Conv2d(192, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
        )
        self.dec1 = nn.Sequential(
            nn.Conv2d(128+96, 96, 3, padding=1, bias=False),
            nn.BatchNorm2d(96), nn.ReLU(inplace=True),
            nn.Conv2d(96, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
        )
        self.dec0 = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
        )
        self.seg_head = nn.Conv2d(32, 1, 1)
        self.clf_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(768, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.4), nn.Linear(256, n_clf_classes),
        )

    def _to_nchw(self, feat):
        """Convertit NHWC → NCHW si nécessaire (Swin retourne NHWC)."""
        if feat.ndim == 4 and feat.shape[1] == feat.shape[2]:
            return feat.permute(0, 3, 1, 2).contiguous()
        return feat

    def forward(self, x):
        x3 = self.chan_adapt(x)
        feats = self.backbone(x3)
        f0 = self._to_nchw(feats[0])  # (B, 96,  56, 56)
        f1 = self._to_nchw(feats[1])  # (B, 192, 28, 28)
        f2 = self._to_nchw(feats[2])  # (B, 384, 14, 14)
        f3 = self._to_nchw(feats[3])  # (B, 768,  7,  7)

        clf_logits = self.clf_head(f3)

        d3 = F.interpolate(f3, size=f2.shape[2:], mode='bilinear', align_corners=False)
        d3 = self.dec3(torch.cat([d3, f2], dim=1))

        d2 = F.interpolate(d3, size=f1.shape[2:], mode='bilinear', align_corners=False)
        d2 = self.dec2(torch.cat([d2, f1], dim=1))

        d1 = F.interpolate(d2, size=f0.shape[2:], mode='bilinear', align_corners=False)
        d1 = self.dec1(torch.cat([d1, f0], dim=1))

        d0 = F.interpolate(d1, size=x.shape[2:], mode='bilinear', align_corners=False)
        d0 = self.dec0(d0)

        return self.seg_head(d0), clf_logits


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n🔢 PARAMÈTRES DES MODÈLES :")
for name, cls, kw in [
    ("UNetClassifier",     UNetClassifier,     {}),
    ("UNetWithClassifier", UNetWithClassifier, {}),
    ("AttentionUNet",      AttentionUNet,      {}),
]:
    m = cls(**kw); print(f"  {name:25s} : {count_parameters(m)/1e6:.2f}M"); del m
try:
    _sw = SwinHAFNet(pretrained=False)
    print(f"  {'SwinHAFNet':25s} : {count_parameters(_sw)/1e6:.2f}M")
    del _sw
except Exception as e:
    print(f"  SwinHAFNet          : ⚠️ {e}")

print("✅ Bloc 4 terminé")

# ============================================================================
# BLOC 5 : LOSS FUNCTIONS ET MÉTRIQUES (IDENTIQUES AU CODE ORIGINAL)
# ============================================================================
print("\n" + "="*70)
print("📉 BLOC 5 : LOSS FUNCTIONS ET MÉTRIQUES")
print("="*70)

class DiceLoss(nn.Module):
    """
    Dice Loss = 1 - Dice Score.
    Dice = 2|X∩Y| / (|X|+|Y|+smooth)
    Gère le déséquilibre fond(98%)/tumeur(2%).
    IDENTIQUE au code Kaggle original.
    """
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, inputs, targets):
        inputs  = torch.sigmoid(inputs)
        inputs  = inputs.view(-1)
        targets = targets.view(-1)
        inter   = (inputs * targets).sum()
        dice    = (2.*inter + self.smooth) / (inputs.sum() + targets.sum() + self.smooth)
        return 1 - dice


class CombinedLoss(nn.Module):
    """
    Loss combinée = BCEWithLogitsLoss(pos_weight) + DiceLoss
    BCE  : supervision pixel-par-pixel (précision locale)
    Dice : supervision overlap global (robustesse au déséquilibre)
    pos_weight : compense déséquilibre fond/tumeur (ici ≈10)
    IDENTIQUE au code Kaggle original.
    """
    def __init__(self, pos_weight=None):
        super().__init__()
        self.bce  = nn.BCEWithLogitsLoss(pos_weight=pos_weight) \
                    if pos_weight is not None else nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
    def forward(self, inputs, targets):
        if (self.bce.pos_weight is not None and
                self.bce.pos_weight.device != inputs.device):
            self.bce.pos_weight = self.bce.pos_weight.to(inputs.device)
        return self.bce(inputs, targets) + self.dice(inputs, targets)


def calculate_seg_metrics(pred_mask, true_mask, threshold=0.5):
    """
    Métriques seg batch-level (tenseurs GPU) — IDENTIQUE au code original.
    Retourne (dice, iou, pixel_accuracy) scalaires.
    """
    pred_b = (pred_mask > threshold).float()
    true_b = (true_mask > 0.5).float()
    pred_f = pred_b.view(-1); true_f = true_b.view(-1)
    inter  = (pred_f * true_f).sum()
    union  = pred_f.sum() + true_f.sum() - inter
    correct= (pred_f == true_f).sum()
    eps    = 1e-6
    dice   = ((2*inter + eps) / (pred_f.sum() + true_f.sum() + eps)).item()
    iou    = ((inter   + eps) / (union + eps)).item()
    px_acc = (correct.float() / pred_f.numel()).item()
    return dice, iou, px_acc


def calculate_seg_metrics_full(pred_list, gt_list, threshold=0.5,
                                 exclude_black_masks=True):
    """
    Métriques COMPLÈTES agrégées (TP/TN/FP/FN sur toute l'époque).
    exclude_black_masks=True : exclut les images no_tumor (GT tout-zéro).
    IDENTIQUE au code original + option exclusion.
    """
    _nan = {k: float('nan') for k in
            ['Dice','IoU','Sensitivity','Specificity','Precision','Pixel_Acc']}
    if not pred_list or not gt_list:
        return _nan
    try:
        preds = torch.cat(pred_list, dim=0)
        gts   = torch.cat(gt_list,   dim=0)
    except Exception:
        return _nan

    if exclude_black_masks:
        gt_sums = gts.view(gts.shape[0], -1).sum(dim=1)
        valid   = gt_sums > 0
        if valid.sum() == 0:
            return _nan
        preds = preds[valid]; gts = gts[valid]

    pf  = (preds > threshold).float().view(-1)
    gf  = (gts   > 0.5      ).float().view(-1)
    TP  = (pf * gf).sum().item()
    TN  = ((1-pf)*(1-gf)).sum().item()
    FP  = (pf*(1-gf)).sum().item()
    FN  = ((1-pf)*gf).sum().item()
    eps = 1e-6
    return {
        'Dice'       : round((2*TP+eps)/(2*TP+FP+FN+eps), 6),
        'IoU'        : round((TP+eps)/(TP+FP+FN+eps),     6),
        'Sensitivity': round((TP+eps)/(TP+FN+eps),         6),
        'Specificity': round((TN+eps)/(TN+FP+eps),         6),
        'Precision'  : round((TP+eps)/(TP+FP+eps),         6),
        'Pixel_Acc'  : round((TP+TN)/(TP+TN+FP+FN+eps),   6),
    }


print("✅ Bloc 5 terminé")

# ============================================================================
# BLOC 6 : DATASETS ET AUGMENTATIONS (IDENTIQUES AU CODE ORIGINAL)
# ============================================================================
print("\n" + "="*70)
print("📦 BLOC 6 : DATASETS ET AUGMENTATIONS")
print("="*70)

def get_seg_transforms(split='train'):
    """
    Augmentations segmentation — IDENTIQUES au code Kaggle original.
    Train : augmentations géométriques + intensité (synchronisées image+masque).
    Val/Test : ToTensorV2 uniquement.
    """
    if split == 'train':
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1),
                     rotate=(-25,25), p=0.7),
            A.ElasticTransform(alpha=1, sigma=50, p=0.3),
            A.GridDistortion(p=0.3),
            A.OpticalDistortion(distort_limit=0.1, p=0.3),
            A.RandomBrightnessContrast(0.3, 0.3, p=0.5),
            A.RandomGamma(gamma_limit=(80,120), p=0.3),
            A.GaussNoise(p=0.3),
            A.GaussianBlur(blur_limit=(3,5), p=0.2),
            ToTensorV2()
        ])
    return A.Compose([ToTensorV2()])


def _npy_to_2d(arr):
    """Assure que le tableau est 2D (H,W) float32."""
    arr = arr.astype(np.float32)
    while arr.ndim > 2:
        arr = arr[...,0] if arr.shape[-1]==1 else arr[0]
    return arr


def _apply_albumentations(transform, img_2d, mask_2d=None):
    """Applique Albumentations sur image 2D + masque 2D optionnel."""
    img_3d = img_2d[:,:,None]   # (H,W,1) pour Albumentations
    if mask_2d is not None:
        msk_3d = mask_2d[:,:,None]
        aug    = transform(image=img_3d, mask=msk_3d)
        img_t  = aug['image'].float()
        msk    = aug['mask']
        if isinstance(msk, torch.Tensor):
            if msk.ndim==3 and msk.shape[-1]==1:
                msk = msk.permute(2,0,1).float()
            elif msk.ndim==2:
                msk = msk.unsqueeze(0).float()
            else:
                msk = msk.float()
        else:
            msk = torch.from_numpy(np.array(msk)[...,0]).unsqueeze(0).float()
        return img_t, msk
    aug = transform(image=img_3d)
    return aug['image'].float(), None


def _infer_label(stem, cls_map):
    """Déduit le label depuis le nom du fichier .npy."""
    _codes = {'_gl_':'glioma','_me_':'meningioma',
              '_no_':'no_tumor','_pi_':'pituitary'}
    sl = stem.lower()
    for code, cls in _codes.items():
        if code in sl: return cls_map.get(cls, 0)
    for cls, idx in cls_map.items():
        if cls in sl: return idx
    return 0


class SegmentationNewDataset(Dataset):
    """
    Dataset segmentation_new : 5000 train + 1000 test (avec no_tumor).
    Retourne (image:(1,H,W), mask:(1,H,W), label:int)
    Masques no_tumor = zeros → exclus du calcul Dice/IoU en inférence.
    """
    def __init__(self, proc_dir, split='train', transform=None):
        super().__init__()
        self.transform = transform
        self.samples   = []  # (img_path, mask_path, label, is_black_mask)

        img_dir  = Path(proc_dir) / 'segmentation_new' / split / 'images'
        mask_dir = Path(proc_dir) / 'segmentation_new' / split / 'masks'
        map_path = Path(proc_dir) / 'class_mapping.json'
        cls_map  = json.load(open(map_path)) if map_path.exists() else CLASS_TO_IDX

        if not img_dir.exists():
            print(f"  ❌ {img_dir} absent !")
            return

        for img_p in sorted(img_dir.glob('*.npy')):
            msk_p = mask_dir / img_p.name
            if not msk_p.exists():
                continue
            label = _infer_label(img_p.stem, cls_map)
            # Détecter si masque noir (no_tumor)
            msk_arr = np.load(msk_p)
            is_black = float(np.sum(msk_arr > 0)) == 0.
            self.samples.append((img_p, msk_p, label, is_black))

        n_real = sum(1 for s in self.samples if not s[3])
        n_blk  = sum(1 for s in self.samples if s[3])
        print(f"  ✅ SegNew [{split}]: {len(self.samples)} total "
              f"({n_real} tumeur + {n_blk} no_tumor/masque_noir)")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_p, msk_p, label, _ = self.samples[idx]
        img  = _npy_to_2d(np.load(img_p))
        mask = _npy_to_2d(np.load(msk_p))
        mask = (mask > 0.5).astype(np.float32)
        if self.transform:
            img_t, msk_t = _apply_albumentations(self.transform, img, mask)
        else:
            img_t = torch.from_numpy(img[:,:,None]).permute(2,0,1).float()
            msk_t = torch.from_numpy(mask).unsqueeze(0).float()
        return img_t, msk_t, torch.tensor(label, dtype=torch.long)


class SegmentationDataset(Dataset):
    """
    Dataset segmentation standard : 3933 train + 860 test (sans no_tumor).
    Retourne (image:(1,H,W), mask:(1,H,W))
    """
    def __init__(self, proc_dir, split='train', transform=None):
        super().__init__()
        self.transform = transform
        self.samples   = []

        img_dir  = Path(proc_dir) / 'segmentation' / split / 'images'
        mask_dir = Path(proc_dir) / 'segmentation' / split / 'masks'

        if img_dir.exists():
            for img_p in sorted(img_dir.glob('*.npy')):
                msk_p = mask_dir / img_p.name
                if msk_p.exists():
                    self.samples.append((img_p, msk_p))

        print(f"  ✅ Seg [{split}]: {len(self.samples)} paires (sans no_tumor)")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_p, msk_p = self.samples[idx]
        img  = _npy_to_2d(np.load(img_p))
        mask = _npy_to_2d(np.load(msk_p))
        mask = (mask > 0.5).astype(np.float32)
        if self.transform:
            img_t, msk_t = _apply_albumentations(self.transform, img, mask)
        else:
            img_t = torch.from_numpy(img[:,:,None]).permute(2,0,1).float()
            msk_t = torch.from_numpy(mask).unsqueeze(0).float()
        return img_t, msk_t


# Test rapide
print("\n🧪 Test des datasets :")
try:
    _d = SegmentationNewDataset(str(PROC_DIR), 'train', get_seg_transforms('test'))
    if len(_d) > 0:
        _i, _m, _l = _d[0]
        print(f"  SegNew train[0]: img={_i.shape} mask={_m.shape} label={_l.item()}")
        assert _i.max() <= 1.01, "Image non normalisée !"
    del _d
    print("  ✅ Datasets OK")
except Exception as e:
    print(f"  ❌ {e}")

# Poids de classes
_clf_counts = {}
for cls in CLASS_NAMES:
    _d = PROC_DIR / 'train' / cls
    _clf_counts[cls] = len(list(_d.glob('*.npy'))) if _d.exists() else 0
_total_clf = sum(_clf_counts.values())
_cw_list   = [_total_clf / (N_CLASSES * max(_clf_counts.get(c, 1), 1))
               for c in CLASS_NAMES]
class_weights = torch.tensor(_cw_list, dtype=torch.float32).to(DEVICE)
print(f"\n⚖️  Poids classes  : {[f'{w:.3f}' for w in _cw_list]}")

pos_weight = torch.tensor([HYPERPARAMS['POS_WEIGHT']], dtype=torch.float32).to(DEVICE)
print(f"⚖️  pos_weight seg : {HYPERPARAMS['POS_WEIGHT']:.1f}")

print("✅ Bloc 6 terminé")

# ============================================================================
# BLOC 7 : CHARGEMENT DES MODÈLES PRÉ-ENTRAÎNÉS (ARCHITECTURES CORRIGÉES)
# ============================================================================
print("\n" + "="*70)
print("📦 BLOC 7 : CHARGEMENT MODÈLES PRÉ-ENTRAÎNÉS")
print("="*70)

def _load_pth_safe(ModelClass, pth_path, name, strict=True, **kwargs):
    """
    Charge un modèle .pth avec architecture corrigée.
    Tentative strict=True, puis strict=False si >80% clés.
    Garantit model.eval() après chargement.
    """
    p = Path(pth_path)
    if not p.exists():
        print(f"  ❌ {name} : {p.name} introuvable")
        return None
    try:
        m  = ModelClass(**kwargs).to(DEVICE)
        try:
            sd = torch.load(str(p), map_location=DEVICE, weights_only=True)
        except TypeError:
            sd = torch.load(str(p), map_location=DEVICE)

        try:
            m.load_state_dict(sd, strict=True)
            m.eval()
            print(f"  ✅ {name:30s} | {count_parameters(m)/1e6:.2f}M | strict=True ✓")
            return m
        except RuntimeError as e1:
            miss, unexp = m.load_state_dict(sd, strict=False)
            pct = (1 - len(miss)/max(len(m.state_dict()), 1)) * 100
            if pct >= 80:
                m.eval()
                print(f"  ✅ {name:30s} | {pct:.0f}% clés | strict=False "
                      f"(miss={len(miss)}, unexp={len(unexp)})")
                return m
            else:
                print(f"  ❌ {name:30s} : {pct:.0f}% clés seulement\n"
                      f"     → {str(e1)[:120]}")
                return None
    except Exception as e:
        print(f"  ❌ {name} : {e}")
        return None


print("\n📦 Chargement modèles pré-entraînés...")

# ── UNetClassifier (architecture CORRIGÉE : Down(512,512), Linear(512,256)) ──
MODEL_CLF = _load_pth_safe(
    UNetClassifier, MODELS_DIR/'unet_clf_best.pth', 'UNetClassifier',
    n_channels=1, n_classes=4)

# ── AttentionUNet ─────────────────────────────────────────────────────────────
MODEL_ATT = _load_pth_safe(
    AttentionUNet, MODELS_DIR/'att_unet_best.pth', 'AttentionUNet',
    n_channels=1, n_classes=1)

# ── UNetWithClassifier original (joint de base, NON joint_total) ─────────────
MODEL_JOINT_ORIG = _load_pth_safe(
    UNetWithClassifier, MODELS_DIR/'unet_joint_best.pth', 'UNetJoint_Orig',
    n_channels=1, n_seg_classes=1, n_clf_classes=4)

# ── UNet Seg ──────────────────────────────────────────────────────────────────
# (On le recrée avec l'architecture originale)
class UNet(nn.Module):
    """UNet standard — IDENTIQUE au code Kaggle original."""
    def __init__(self, n_channels=1, n_classes=1, bilinear=False):
        super().__init__()
        factor = 2 if bilinear else 1
        self.inc   = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128); self.down2 = Down(128, 256)
        self.down3 = Down(256, 512); self.down4 = Down(512, 1024//factor)
        self.up1   = Up(1024, 512//factor, bilinear); self.up2 = Up(512, 256//factor, bilinear)
        self.up3   = Up(256, 128//factor, bilinear);  self.up4 = Up(128, 64, bilinear)
        self.outc  = OutConv(64, n_classes)
    def forward(self, x):
        x1=self.inc(x); x2=self.down1(x1); x3=self.down2(x2)
        x4=self.down3(x3); x5=self.down4(x4)
        x=self.up1(x5,x4); x=self.up2(x,x3); x=self.up3(x,x2); x=self.up4(x,x1)
        return self.outc(x)

MODEL_SEG = _load_pth_safe(
    UNet, MODELS_DIR/'unet_seg_best.pth', 'UNet_Seg',
    n_channels=1, n_classes=1)

n_loaded = sum(1 for m in [MODEL_CLF, MODEL_ATT, MODEL_JOINT_ORIG, MODEL_SEG]
               if m is not None)
print(f"\n  {n_loaded}/4 modèles chargés depuis result_model ✅")
print("✅ Bloc 7 terminé")

# ============================================================================
# BLOC 8 : UTILITAIRES D'ENTRAÎNEMENT ET VISUALISATION
# ============================================================================
print("\n" + "="*70)
print("🛠️  BLOC 8 : UTILITAIRES ENTRAÎNEMENT + VISUALISATION")
print("="*70)

class EarlyStopping:
    """
    Arrêt anticipé avec sauvegarde automatique du meilleur modèle.
    IDENTIQUE au code Kaggle original.
    """
    def __init__(self, patience=15, mode='max', save_path=None):
        self.patience = patience; self.mode = mode; self.save_path = save_path
        self.best = float('-inf') if mode=='max' else float('inf')
        self.counter = 0; self.stop = False

    def __call__(self, score, model=None):
        improved = ((self.mode=='max' and score > self.best) or
                    (self.mode=='min' and score < self.best))
        if improved:
            self.best = score; self.counter = 0
            if self.save_path and model is not None:
                Path(self.save_path).parent.mkdir(parents=True, exist_ok=True)
                torch.save(model.state_dict(), self.save_path)
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


def get_model_summary_table(model, input_size=(1,1,224,224), model_name="Model"):
    """
    Tableau résumé du modèle (Layer | Output Shape | Parameters).
    IDENTIQUE au code Kaggle original.
    """
    data = []; hooks = []

    def _hook(module, inp, out):
        n = sum(p.numel() for p in module.parameters())
        s = str(list(out.shape)) if hasattr(out,'shape') else 'N/A'
        data.append({'Layer':module.__class__.__name__,
                      'Output Shape':s, 'Parameters':f"{n:,}"})

    for _, m in model.named_modules():
        if len(list(m.children())) == 0:
            hooks.append(m.register_forward_hook(_hook))

    model.eval()
    dummy = torch.randn(*input_size).to(DEVICE)
    try:
        with torch.no_grad():
            with _amp(): _ = model(dummy)
    except Exception: pass
    finally:
        for h in hooks: h.remove()
    del dummy

    df         = pd.DataFrame(data)
    total_p    = sum(p.numel() for p in model.parameters())
    trainable_p= sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n  ┌─ {model_name}")
    print(f"  │  Total params     : {total_p:,}")
    print(f"  │  Trainable params : {trainable_p:,}")
    print(f"  └─ Leaf layers      : {len(data)}")
    if len(df) > 0:
        print(df.head(8).to_string(index=False))
        if len(df) > 9: print(f"  ... ({len(df)-9} autres couches) ...")
    return df, total_p


def plot_training_curves(history, model_name, save_path):
    """
    Courbe Loss + Métriques côte à côte.
    IDENTIQUE au code Kaggle original.
    """
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for key, lbl, col in [('train_loss','Train Loss','#e74c3c'),
                            ('val_loss',  'Val Loss',  '#3498db'),
                            ('test_loss', 'Test Loss', '#2ecc71')]:
        if key in history and history[key]:
            axes[0].plot(history[key], label=lbl, color=col, lw=2)
    axes[0].set_title(f'{model_name} — Loss (Train/Val/Test)',
                      fontweight='bold', fontsize=14)
    axes[0].set_xlabel('Époque'); axes[0].set_ylabel('Loss')
    axes[0].legend(fontsize=10); axes[0].grid(alpha=0.3)

    metric_cfg = [('val_acc','Val Acc','#9b59b6','-'),
                  ('val_iou','Val IoU','#f39c12','-'),
                  ('val_dice','Val Dice','#8e44ad','-.'),
                  ('val_pixel_acc','Val PxAcc','#16a085',':')]
    for key, lbl, col, ls in metric_cfg:
        if key in history and history[key]:
            axes[1].plot(history[key], label=lbl, color=col, ls=ls, lw=2)
    axes[1].set_title(f'{model_name} — Métriques (Val)',
                      fontweight='bold', fontsize=14)
    axes[1].set_xlabel('Époque'); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

    plt.suptitle(model_name, fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close(fig)
    print(f"  📊 Courbe → {Path(save_path).name}")


def plot_metric_curve(history, model_name, metric_key, save_path,
                       ylabel=None, title=None):
    """
    Courbes Train/Val/Test séparées pour une métrique.
    IDENTIQUE au code Kaggle original.
    """
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = {'train':'#2196F3','val':'#FF9800','test':'#F44336'}
    styles = {'train':'-','val':'-','test':'--'}
    plotted = False
    for split in ['train','val','test']:
        key  = f'{split}_{metric_key}'
        data = history.get(key, [])
        if data:
            ax.plot(range(1,len(data)+1), data, color=colors[split], lw=2.5,
                    ls=styles[split], label=split.capitalize(), alpha=0.9,
                    marker='o', ms=3)
            plotted = True
    if not plotted:
        plt.close(fig); return
    ax.set_xlabel('Époque', fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel or metric_key, fontsize=12, fontweight='bold')
    ax.set_title(title or f'{model_name} — {metric_key}',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=11); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close(fig)
    print(f"  📊 {Path(save_path).name}")


def plot_confusion_matrix(cm, class_names, title, save_path):
    """Heatmap matrice de confusion. IDENTIQUE au code Kaggle original."""
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Prédit',  fontweight='bold', fontsize=12)
    ax.set_ylabel('Réel',    fontweight='bold', fontsize=12)
    ax.set_title(title,      fontweight='bold', fontsize=13)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close(fig)
    print(f"  💾 Matrice → {Path(save_path).name}")


def plot_9_predictions(results_list, model_name, save_path, n_show=9):
    """
    Visualise 9 images avec image originale, masque GT, masque prédit, superposition.
    SIMILAIRE au code original UNet (affichage 4 colonnes × n_show lignes).
    Légende : Vert=TP | Rouge=FP | Bleu=FN
    """
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)

    # Choisir des exemples représentatifs (avec vraie tumeur + bon Dice)
    valid = [r for r in results_list
             if r.get('gt_mask') is not None and r.get('seg_mask') is not None
             and not r.get('gt_is_black', False)]
    if not valid:
        print(f"  ⚠️  Aucune prédiction disponible pour {model_name}")
        return

    # Trier par Dice décroissant pour montrer les meilleurs + quelques mauvais
    valid.sort(key=lambda r: r.get('dice',0), reverse=True)
    # Sélectionner : 6 meilleurs + 3 moins bons
    show = (valid[:6] + valid[-3:])[:n_show]
    n    = len(show)

    fig, axes = plt.subplots(n, 4, figsize=(20, n * 4.5))
    if n == 1: axes = axes.reshape(1, -1)

    patches = [mpatches.Patch(color='green',  label='TP (correct)'),
               mpatches.Patch(color='red',    label='FP (faux +)'),
               mpatches.Patch(color='blue',   label='FN (manqué)')]

    for i, r in enumerate(show):
        img  = r['img_arr']
        gt   = r['gt_mask']
        pred = r['seg_mask']
        dice = r.get('dice', 0.)
        iou  = r.get('iou',  0.)
        cls  = r.get('true_class', '?')
        pred_cls = r.get('pred_class', '?')

        # Overlay RGB
        tp = ((pred > 0.5) & (gt > 0.5)).astype(np.float32)
        fp = ((pred > 0.5) & (gt < 0.5)).astype(np.float32)
        fn = ((pred < 0.5) & (gt > 0.5)).astype(np.float32)

        overlay = np.stack([img]*3, axis=-1)
        overlay[:,:,1] = np.clip(overlay[:,:,1] + tp*0.7, 0, 1)  # vert TP
        overlay[:,:,0] = np.clip(overlay[:,:,0] + fp*0.7, 0, 1)  # rouge FP
        overlay[:,:,2] = np.clip(overlay[:,:,2] + fn*0.7, 0, 1)  # bleu FN

        axes[i,0].imshow(img, cmap='gray')
        axes[i,0].set_title(f"Image\n{r.get('img_name','?')[:18]}\nVrai:{cls}", fontsize=8)
        axes[i,0].axis('off')

        axes[i,1].imshow(gt, cmap='hot')
        axes[i,1].set_title("GT Masque", fontsize=8)
        axes[i,1].axis('off')

        axes[i,2].imshow(pred, cmap='hot')
        axes[i,2].set_title(f"Masque Prédit\nPrédit:{pred_cls}", fontsize=8)
        axes[i,2].axis('off')

        axes[i,3].imshow(np.clip(overlay, 0, 1))
        axes[i,3].set_title(f"Superposition\nDice={dice:.3f} IoU={iou:.3f}", fontsize=8)
        axes[i,3].axis('off')

    plt.suptitle(f'{model_name} — 9 Prédictions (Vert=TP | Rouge=FP | Bleu=FN)',
                 fontsize=13, fontweight='bold')
    fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=9,
               bbox_to_anchor=(0.5, -0.01))
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches='tight'); plt.close(fig)
    print(f"  💾 9 prédictions → {Path(save_path).name}")


def plot_all_superpositions(results_list, model_name, out_dir,
                             batch_size=50, max_images=None):
    """
    Superposition de TOUTES les images (860 ou 1000) avec masques prédits.
    Affiche pour chaque image :
      - Image prétraitée originale
      - Masque prédit superposé (TP=vert, FP=rouge, FN=bleu)
      - Valeur Dice et IoU
    Génère des planches de batch_size images.
    ✅ CORRECTION FileNotFoundError : mkdir avant chaque sauvegarde.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)  # ← FIX FileNotFoundError

    # Filtrer les images avec masque GT (exclure no_tumor)
    seg_r = [r for r in results_list
             if r.get('gt_mask') is not None
             and r.get('seg_mask') is not None
             and not r.get('gt_is_black', False)]
    if not seg_r: print(f"  ⚠️  Pas de résultats avec masques GT"); return
    if max_images: seg_r = seg_r[:max_images]

    print(f"\n  🖼️  {len(seg_r)} images → planches de {batch_size}")
    n_batches = math.ceil(len(seg_r) / batch_size)

    for b in range(n_batches):
        batch = seg_r[b*batch_size : (b+1)*batch_size]
        n_show = len(batch)
        cols = min(10, n_show)
        rows = math.ceil(n_show / cols)

        fig, axes = plt.subplots(rows, cols, figsize=(cols*2.5, rows*3))
        axes = np.array(axes).flatten()

        for i, r in enumerate(batch):
            ax   = axes[i]
            img  = r['img_arr']
            gt   = r['gt_mask']
            pred = r['seg_mask']
            dice = r.get('dice', 0.)
            iou  = r.get('iou',  0.)

            # Superposition : image + masque prédit colorisé
            ov = np.stack([img]*3, axis=-1)
            ov = np.clip(ov, 0, 1)
            ov[(pred>0.5)&(gt>0.5)] = [0., 0.9, 0.]   # TP vert
            ov[(pred>0.5)&(gt<0.5)] = [0.9, 0., 0.]   # FP rouge
            ov[(pred<0.5)&(gt>0.5)] = [0., 0., 0.9]   # FN bleu

            ax.imshow(np.clip(ov, 0, 1))
            ax.set_title(
                f"{(r.get('true_class','?'))[:3]}\n"
                f"D={dice:.2f} I={iou:.2f}",
                fontsize=5.5, pad=1
            )
            ax.axis('off')

        for i in range(n_show, len(axes)):
            axes[i].axis('off')

        plt.suptitle(
            f"{model_name} — Planche {b+1}/{n_batches}  "
            f"[Vert=TP | Rouge=FP | Bleu=FN]",
            fontsize=8, fontweight='bold'
        )
        plt.tight_layout(rect=[0, 0, 1, 0.96])

        _path = out_dir / f'superposition_{model_name}_batch{b+1:03d}.png'
        plt.savefig(_path, dpi=100, bbox_inches='tight'); plt.close(fig)
        print(f"    💾 Planche {b+1}/{n_batches} → {_path.name}")

    print(f"  ✅ {n_batches} planches sauvegardées → {out_dir}")


def _get_last(hist, key):
    d = hist.get(key, [])
    return round(float(d[-1]), 4) if d else float('nan')

def _get_best(hist, key, mode='max'):
    d = hist.get(key, [])
    if not d: return float('nan')
    return round(float(max(d) if mode=='max' else min(d)), 4)


print("✅ Bloc 8 terminé")

# ============================================================================
# BLOC 9 : ENTRAÎNEMENT JOINT_TOTAL (5000 images avec no_tumor)
# ============================================================================
print("\n" + "="*80)
print("🎓 BLOC 9 : ENTRAÎNEMENT JOINT_TOTAL")
print("="*80)
"""
joint_total = UNetWithClassifier entraîné sur segmentation_new (5000 images).
  • no_tumor inclus avec masques noirs → support≠0 dans rapport clf ✅
  • Split 80/20 : 4000 train | 1000 val | 1000 test fixe
  • Score combiné = (IoU_val + Acc_val) / 2 → critère early stopping
  • ReduceLROnPlateau : patience=5, factor=0.5 (sans verbose — ✅ PyTorch ≥2.2)
  • Checkpoint toutes les 10 époques → résistance à l'arrêt GPU Kaggle
  • LR = 5e-4 (corrigé — était 1e-3, trop élevé → mauvaise convergence)
"""

# ── Dossiers de résultats joint ───────────────────────────────────────────────
JOINT_RESULTS = RESULTS_DIR / 'joint_total'
JOINT_PLOTS   = JOINT_RESULTS / 'plots'
JOINT_INF_DIR = JOINT_RESULTS / 'inference'
for _d in [JOINT_RESULTS, JOINT_PLOTS, JOINT_INF_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# ── Datasets ──────────────────────────────────────────────────────────────────
print("\n📦 Chargement datasets joint_total...")
full_jnt_ds  = SegmentationNewDataset(str(PROC_DIR), 'train',
                                       get_seg_transforms('train'))
test_jnt_ds  = SegmentationNewDataset(str(PROC_DIR), 'test',
                                       get_seg_transforms('test'))

if len(full_jnt_ds) == 0:
    raise ValueError("Dataset joint_total vide ! Vérifiez PROC_DIR/segmentation_new/")

_ts_j = int(0.8 * len(full_jnt_ds)); _vs_j = len(full_jnt_ds) - _ts_j
train_jnt_sub, val_jnt_sub = random_split(
    full_jnt_ds, [_ts_j, _vs_j],
    generator=torch.Generator().manual_seed(42))

print(f"  Train:{len(train_jnt_sub)} | Val:{len(val_jnt_sub)} | Test:{len(test_jnt_ds)}")

# ── DataLoaders ───────────────────────────────────────────────────────────────
_nw_jnt = 2 if torch.cuda.is_available() else 0
_pm_jnt = torch.cuda.is_available()
_bs_j   = HYPERPARAMS['BATCH_JOINT']

def _dl_kwargs(train=True):
    kw = dict(num_workers=_nw_jnt if train else max(_nw_jnt//2,0),
              pin_memory=_pm_jnt,
              persistent_workers=(True if train and _nw_jnt>0 else False))
    if train and _nw_jnt > 0:
        kw['prefetch_factor'] = 2
    return kw

train_jnt_loader = DataLoader(train_jnt_sub, batch_size=_bs_j,
                               shuffle=True, **_dl_kwargs(True))
val_jnt_loader   = DataLoader(val_jnt_sub,   batch_size=_bs_j,
                               shuffle=False, **_dl_kwargs(False))
test_jnt_loader  = DataLoader(test_jnt_ds,   batch_size=_bs_j,
                               shuffle=False, **_dl_kwargs(False))

# ── Modèle ────────────────────────────────────────────────────────────────────
print("\n🧠 Initialisation UNetWithClassifier (joint_total)...")
model_joint   = UNetWithClassifier(n_channels=1, n_seg_classes=1, n_clf_classes=4).to(DEVICE)
_jnt_pth      = MODELS_DIR / 'joint_total.pth'
_jnt_ckpt_dir = MODELS_DIR / 'checkpoints_joint'
_jnt_ckpt_dir.mkdir(parents=True, exist_ok=True)

_, total_params_jnt = get_model_summary_table(
    model_joint, (1,1,224,224), "UNetWithClassifier (joint_total)")

# ── Reprendre depuis checkpoint si disponible (résistance arrêt Kaggle) ───────
_start_epoch = 1
for _ckpt in sorted(_jnt_ckpt_dir.glob('checkpoint_ep*.pth'), reverse=True):
    try:
        model_joint.load_state_dict(
            torch.load(str(_ckpt), map_location=DEVICE))
        _ep_str = _ckpt.stem.replace('checkpoint_ep', '')
        _start_epoch = int(_ep_str) + 1
        print(f"  ♻️  Reprise depuis checkpoint : {_ckpt.name} (epoch {_start_epoch})")
        break
    except Exception:
        continue

optimizer_jnt = optim.AdamW(model_joint.parameters(),
                              lr=HYPERPARAMS['LR_JOINT'],
                              weight_decay=HYPERPARAMS['WEIGHT_DECAY'])
# ReduceLROnPlateau sans verbose (PyTorch ≥2.2 : verbose supprimé)
scheduler_jnt = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_jnt, mode='max', patience=5, factor=0.5, min_lr=1e-6)
crit_seg_jnt  = CombinedLoss(pos_weight=pos_weight)
crit_clf_jnt  = nn.CrossEntropyLoss(weight=class_weights)
scaler_jnt    = GradScaler() if USE_AMP else None

early_stop_jnt = EarlyStopping(patience=HYPERPARAMS['EARLY_STOP_PATIENCE'],
                                 mode='max', save_path=str(_jnt_pth))

history_jnt = {f'{s}_{m}': []
               for s in ['train','val','test']
               for m in ['loss','acc','f1','iou','dice','pixel_acc']}

t_train_jnt = t_val_jnt = t_test_jnt = 0.
best_jnt_preds = best_jnt_labels = []
best_jnt_seg   = {}
epochs_done_jnt = 0

print(f"  Params : {total_params_jnt:,} ({total_params_jnt/1e6:.2f}M)")
print(f"  LR     : {HYPERPARAMS['LR_JOINT']}")
print(f"  Epochs : {HYPERPARAMS['EPOCHS']} max | "
      f"Early stop patience={HYPERPARAMS['EARLY_STOP_PATIENCE']}")
print("\n🏋️  Entraînement (100 époques max)...")

for epoch in range(_start_epoch, HYPERPARAMS['EPOCHS'] + 1):

    # ── TRAIN ──────────────────────────────────────────────────────────────────
    model_joint.train()
    tr_loss = tr_correct = tr_total = 0
    _tr_sp, _tr_sg, _tr_cp, _tr_cl = [], [], [], []
    _t0 = time.time()

    for imgs, masks, lbls in tqdm(train_jnt_loader,
                                   desc=f"  Ep {epoch:3d} Train", leave=False):
        imgs, masks, lbls = imgs.to(DEVICE), masks.to(DEVICE), lbls.to(DEVICE)
        optimizer_jnt.zero_grad(set_to_none=True)

        if USE_AMP:
            with autocast():
                seg_out, clf_out = model_joint(imgs)
                loss = (HYPERPARAMS['SEG_LOSS_W'] * crit_seg_jnt(seg_out, masks) +
                        HYPERPARAMS['CLF_LOSS_W'] * crit_clf_jnt(clf_out, lbls))
            scaler_jnt.scale(loss).backward()
            scaler_jnt.unscale_(optimizer_jnt)
            nn.utils.clip_grad_norm_(model_joint.parameters(), 1.0)
            scaler_jnt.step(optimizer_jnt); scaler_jnt.update()
        else:
            seg_out, clf_out = model_joint(imgs)
            loss = (HYPERPARAMS['SEG_LOSS_W'] * crit_seg_jnt(seg_out, masks) +
                    HYPERPARAMS['CLF_LOSS_W'] * crit_clf_jnt(clf_out, lbls))
            loss.backward()
            nn.utils.clip_grad_norm_(model_joint.parameters(), 1.0)
            optimizer_jnt.step()

        tr_loss += loss.item()
        _, pred  = torch.max(clf_out, 1)
        tr_correct += (pred==lbls).sum().item()
        tr_total   += lbls.size(0)
        _tr_cp.extend(pred.cpu().numpy().tolist())
        _tr_cl.extend(lbls.cpu().numpy().tolist())
        with torch.no_grad():
            _tr_sp.append(torch.sigmoid(seg_out.detach()).cpu())
            _tr_sg.append(masks.cpu())

    t_train_jnt += time.time() - _t0
    tr_loss /= len(train_jnt_loader)
    tr_acc   = tr_correct / max(tr_total, 1)
    tr_f1    = f1_score(_tr_cl, _tr_cp, labels=CLF_LABELS, average='weighted', zero_division=0)
    tr_seg   = calculate_seg_metrics_full(_tr_sp, _tr_sg, exclude_black_masks=True)

    # ── VALIDATION ─────────────────────────────────────────────────────────────
    model_joint.eval()
    v_loss = v_correct = v_total = 0
    _v_sp, _v_sg, _v_cp, _v_cl = [], [], [], []
    _t0 = time.time()

    with torch.no_grad():
        for imgs, masks, lbls in val_jnt_loader:
            imgs, masks, lbls = imgs.to(DEVICE), masks.to(DEVICE), lbls.to(DEVICE)
            with _amp():
                seg_out, clf_out = model_joint(imgs)
                loss = (HYPERPARAMS['SEG_LOSS_W'] * crit_seg_jnt(seg_out, masks) +
                        HYPERPARAMS['CLF_LOSS_W'] * crit_clf_jnt(clf_out, lbls))
            v_loss += loss.item()
            _, pred  = torch.max(clf_out, 1)
            v_correct += (pred==lbls).sum().item()
            v_total   += lbls.size(0)
            _v_cp.extend(pred.cpu().numpy().tolist())
            _v_cl.extend(lbls.cpu().numpy().tolist())
            _v_sp.append(torch.sigmoid(seg_out).cpu())
            _v_sg.append(masks.cpu())

    t_val_jnt += time.time() - _t0
    v_loss /= len(val_jnt_loader)
    v_acc   = v_correct / max(v_total, 1)
    v_f1    = f1_score(_v_cl, _v_cp, labels=CLF_LABELS, average='weighted', zero_division=0)
    v_seg   = calculate_seg_metrics_full(_v_sp, _v_sg, exclude_black_masks=True)
    # Pixel accuracy (sur sous-ensemble pour perf)
    _cat_sp = torch.cat(_v_sp[:10]); _cat_sg = torch.cat(_v_sg[:10])
    v_px = float(np.mean([calculate_seg_metrics(
        _cat_sp[i:i+1], _cat_sg[i:i+1])[2]
        for i in range(min(50, len(_cat_sp)))]))

    # ── TEST (monitoring) ──────────────────────────────────────────────────────
    te_loss = te_correct = te_total = 0
    _te_sp, _te_sg, _te_cp, _te_cl = [], [], [], []
    _t0 = time.time()

    with torch.no_grad():
        for imgs, masks, lbls in test_jnt_loader:
            imgs, masks, lbls = imgs.to(DEVICE), masks.to(DEVICE), lbls.to(DEVICE)
            with _amp():
                seg_out, clf_out = model_joint(imgs)
                loss = (HYPERPARAMS['SEG_LOSS_W'] * crit_seg_jnt(seg_out, masks) +
                        HYPERPARAMS['CLF_LOSS_W'] * crit_clf_jnt(clf_out, lbls))
            te_loss += loss.item()
            _, pred   = torch.max(clf_out, 1)
            te_correct += (pred==lbls).sum().item()
            te_total   += lbls.size(0)
            _te_cp.extend(pred.cpu().numpy().tolist())
            _te_cl.extend(lbls.cpu().numpy().tolist())
            _te_sp.append(torch.sigmoid(seg_out).cpu())
            _te_sg.append(masks.cpu())

    t_test_jnt += time.time() - _t0
    te_loss /= max(len(test_jnt_loader), 1)
    te_acc   = te_correct / max(te_total, 1)
    te_f1    = f1_score(_te_cl, _te_cp, labels=CLF_LABELS, average='weighted', zero_division=0)
    te_seg   = calculate_seg_metrics_full(_te_sp, _te_sg, exclude_black_masks=True)

    # ── Historique (identique structure code original) ─────────────────────────
    for split, (l_, a_, f_, seg_, px_) in [
        ('train', (tr_loss, tr_acc, tr_f1, tr_seg, 0.)),
        ('val',   (v_loss,  v_acc,  v_f1,  v_seg,  v_px)),
        ('test',  (te_loss, te_acc, te_f1,  te_seg, 0.)),
    ]:
        history_jnt[f'{split}_loss'     ].append(l_)
        history_jnt[f'{split}_acc'      ].append(a_)
        history_jnt[f'{split}_f1'       ].append(f_)
        history_jnt[f'{split}_iou'      ].append(seg_.get('IoU',  0.))
        history_jnt[f'{split}_dice'     ].append(seg_.get('Dice', 0.))
        history_jnt[f'{split}_pixel_acc'].append(px_ if split=='val' else 0.)

    # Score combiné pour early stopping
    combined = 0.5 * v_seg.get('IoU', 0.) + 0.5 * v_acc
    scheduler_jnt.step(combined)

    print(f"\n  Ep {epoch:3d} | Train Loss:{tr_loss:.4f} Acc:{tr_acc:.4f} "
          f"F1:{tr_f1:.4f} IoU:{tr_seg.get('IoU',0):.4f} "
          f"Dice:{tr_seg.get('Dice',0):.4f}")
    print(f"         | Val   Loss:{v_loss:.4f} Acc:{v_acc:.4f}  "
          f"F1:{v_f1:.4f}  IoU:{v_seg.get('IoU',0):.4f}  "
          f"Dice:{v_seg.get('Dice',0):.4f} PxAcc:{v_px:.4f}")
    print(f"         | Test  Loss:{te_loss:.4f} Acc:{te_acc:.4f} "
          f"F1:{te_f1:.4f} IoU:{te_seg.get('IoU',0):.4f} "
          f"Dice:{te_seg.get('Dice',0):.4f}")
    print(f"         | Combined:{combined:.4f} "
          f"LR:{optimizer_jnt.param_groups[0]['lr']:.2e}")

    # Sauvegarder meilleures prédictions test
    if combined > getattr(early_stop_jnt, '_best_combined', 0.):
        early_stop_jnt._best_combined = combined
        best_jnt_preds  = list(_te_cp)
        best_jnt_labels = list(_te_cl)
        best_jnt_seg    = te_seg

    # Checkpoint toutes les 10 époques (résistance à l'arrêt GPU Kaggle)
    if epoch % 10 == 0:
        _ckpt_path = _jnt_ckpt_dir / f'checkpoint_ep{epoch:03d}.pth'
        torch.save(model_joint.state_dict(), _ckpt_path)
        # Sauvegarder aussi l'historique
        pd.DataFrame(history_jnt).to_csv(
            _jnt_ckpt_dir / f'history_ep{epoch:03d}.csv', index=False)
        print(f"  💾 Checkpoint → {_ckpt_path.name}")

    if early_stop_jnt(combined, model_joint):
        print(f"\n  ⚠️  Early stopping à l'époque {epoch}")
        epochs_done_jnt = epoch; break

    epochs_done_jnt = epoch

if DEVICE.type == 'cuda': torch.cuda.empty_cache()

print(f"\n✅ Joint_Total terminé — {epochs_done_jnt} époques | "
      f"Best Combined: {early_stop_jnt.best:.4f}")

# ── Charger le meilleur checkpoint ────────────────────────────────────────────
if _jnt_pth.exists():
    model_joint.load_state_dict(torch.load(str(_jnt_pth), map_location=DEVICE))
    print(f"  ✅ Meilleur modèle rechargé : {_jnt_pth.name}")
model_joint.eval()

# ── Courbes d'entraînement (IDENTIQUES au code original) ──────────────────────
print("\n📊 Génération des courbes Joint_Total...")
plot_training_curves(history_jnt, 'Joint_Total (5000 img)',
                     JOINT_PLOTS / 'joint_total_training_curves.png')
for _mk, _yl in [('loss','Loss'),('iou','IoU'),('dice','Dice'),
                  ('f1','F1-Score'),('acc','Accuracy')]:
    plot_metric_curve(
        history_jnt, 'Joint_Total', _mk,
        JOINT_PLOTS / f'joint_total_{_mk}_train_val_test.png',
        ylabel=_yl, title=f'Joint_Total — {_yl} (Train/Val/Test)')

# Sauvegarder historique CSV
pd.DataFrame(history_jnt).to_csv(
    JOINT_RESULTS / 'joint_total_history.csv', index=False)

# ── Évaluation finale Joint_Total ─────────────────────────────────────────────
print("\n📊 ÉVALUATION FINALE Joint_Total sur test (1000 images, avec no_tumor) :")
model_joint.eval()
_jnt_fp, _jnt_fl, _jnt_sp, _jnt_sg = [], [], [], []
te_dice_j = te_iou_j = te_px_j = te_n = 0.

with torch.no_grad():
    for imgs, masks, lbls in tqdm(test_jnt_loader, desc="  Final Joint"):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        with _amp():
            seg_out, clf_out = model_joint(imgs)
        _, pred = torch.max(clf_out, 1)
        _jnt_fp.extend(pred.cpu().numpy().tolist())
        _jnt_fl.extend(lbls.numpy().tolist())
        d, i, p = calculate_seg_metrics(torch.sigmoid(seg_out), masks)
        te_dice_j += d; te_iou_j += i; te_px_j += p; te_n += 1
        _jnt_sp.append(torch.sigmoid(seg_out).cpu())
        _jnt_sg.append(masks.cpu())

te_dice_j /= max(te_n, 1); te_iou_j /= max(te_n, 1); te_px_j /= max(te_n, 1)
_jnt_full  = calculate_seg_metrics_full(_jnt_sp, _jnt_sg, exclude_black_masks=True)

print(f"\n  === MÉTRIQUES FINALES Joint_Total (test 1000 images) ===")
print(f"  Clf Accuracy : {(np.array(_jnt_fp)==np.array(_jnt_fl)).mean():.4f}")
print(f"  Seg IoU      : {_jnt_full.get('IoU', te_iou_j):.4f}")
print(f"  Seg Dice     : {_jnt_full.get('Dice',te_dice_j):.4f}")
print(f"  Sensitivity  : {_jnt_full.get('Sensitivity',float('nan')):.4f}")
print(f"  Specificity  : {_jnt_full.get('Specificity',float('nan')):.4f}")
print(f"  Precision    : {_jnt_full.get('Precision',  float('nan')):.4f}")
print(f"  Pixel_Acc    : {_jnt_full.get('Pixel_Acc',  te_px_j):.4f}")

# Rapport de classification (montre le comportement sur no_tumor)
print("\n📋 Classification Report — Joint_Total (avec no_tumor) :")
_jnt_fl_str = [CLASS_NAMES[l] for l in _jnt_fl]
_jnt_fp_str = [CLASS_NAMES[p] for p in _jnt_fp]
print(classification_report(
    _jnt_fl_str, _jnt_fp_str,
    labels=CLASS_NAMES, target_names=CLASS_NAMES, zero_division=0))

# Matrice de confusion Joint_Total
_cm_jnt = confusion_matrix(_jnt_fl, _jnt_fp, labels=list(range(N_CLASSES)))
plot_confusion_matrix(
    _cm_jnt, CLASS_NAMES,
    f'Joint_Total — Test 1000\n(no_tumor visible dans la matrice)',
    JOINT_PLOTS / 'joint_total_confusion_matrix.png')

# Sauvegarder tableau métriques
pd.DataFrame([{
    'Modele'      : 'Joint_Total',
    'N_images'    : 1000,
    'Accuracy'    : round((np.array(_jnt_fp)==np.array(_jnt_fl)).mean(), 4),
    'IoU'         : round(_jnt_full.get('IoU',        float('nan')), 4),
    'Dice'        : round(_jnt_full.get('Dice',       float('nan')), 4),
    'Sensitivity' : round(_jnt_full.get('Sensitivity',float('nan')), 4),
    'Specificity' : round(_jnt_full.get('Specificity',float('nan')), 4),
    'Precision'   : round(_jnt_full.get('Precision',  float('nan')), 4),
    'Pixel_Acc'   : round(_jnt_full.get('Pixel_Acc',  float('nan')), 4),
    'Epochs'      : epochs_done_jnt,
    'Params_M'    : round(total_params_jnt/1e6, 2),
    'Train_Time_s': round(t_train_jnt, 1),
}]).to_csv(JOINT_RESULTS / 'joint_total_metrics.csv', index=False)

# Stocker globaux
results_all['joint_total_clf'] = classification_report(
    _jnt_fl_str, _jnt_fp_str, labels=CLASS_NAMES,
    target_names=CLASS_NAMES, zero_division=0, output_dict=True)
results_all['joint_total_seg'] = _jnt_full
all_histories['Joint_Total']   = history_jnt
complexity_info['Joint_Total'] = {
    'Params (M)'   : round(total_params_jnt/1e6, 3),
    'Train_Time(s)': round(t_train_jnt, 1),
    'Val_Time(s)'  : round(t_val_jnt,   1),
    'Test_Time(s)' : round(t_test_jnt,  1),
}

print("✅ Bloc 9 terminé")

# ============================================================================
# BLOC 10 : INFÉRENCE JOINT_TOTAL (860 + 1000 images)
# ============================================================================
"""
CORRECTION DU CALCUL DE DICE EN INFÉRENCE :
  Entraînement : Dice AGRÉGÉ = 2*ΣTP / (2*ΣTP+ΣFP+ΣFN)
                 → grandes tumeurs dominent → Dice plus élevé (~0.87)
  Inférence    : Dice PAR IMAGE = mean(Dice_i par image)
                 → chaque image pèse pareil → petites tumeurs mal détectées
                   font baisser la moyenne (~0.82)
  Δ typique = 0.03–0.07 → NORMAL et attendu.

Les deux métriques sont calculées et affichées dans ce code.
Images no_tumor (GT tout-zéro) EXCLUES du calcul Dice/IoU.
"""
print("\n" + "="*80)
print("🔬 BLOC 10 : INFÉRENCE JOINT_TOTAL")
print("="*80)

def _load_items(img_dir, mask_dir, source_name):
    """Charge des images .npy pour l'inférence. Retourne liste de dicts."""
    items = []
    img_dir  = Path(img_dir)
    mask_dir = Path(mask_dir) if mask_dir else None
    cls_map  = json.load(open(PROC_DIR/'class_mapping.json')) \
               if (PROC_DIR/'class_mapping.json').exists() else CLASS_TO_IDX

    if not img_dir.exists():
        print(f"  ⚠️  {img_dir} absent"); return items

    for img_p in sorted(img_dir.glob('*.npy')):
        arr = np.load(img_p).astype(np.float32)
        while arr.ndim > 2:
            arr = arr[...,0] if arr.shape[-1]==1 else arr[0]
        if arr.max() > 1.01: arr = arr / 255.0
        arr = np.clip(arr, 0., 1.)

        gt  = None; gt_is_black = False
        if mask_dir is not None:
            mp = mask_dir / img_p.name
            if mp.exists():
                m = np.load(mp).astype(np.float32)
                while m.ndim > 2:
                    m = m[...,0] if m.shape[-1]==1 else m[0]
                m = (m > 0.5).astype(np.float32)
                gt = m; gt_is_black = float(np.sum(m)) == 0.

        label    = _infer_label(img_p.stem, cls_map)
        cls_name = CLASS_NAMES[label] if isinstance(label,int) else str(label)

        items.append({'img_name':img_p.stem, 'img_arr':arr, 'gt_mask':gt,
                       'has_mask': gt is not None, 'gt_is_black':gt_is_black,
                       'true_class':cls_name, 'label_int':label,
                       'source':source_name})
    return items


# ── Listes d'images ────────────────────────────────────────────────────────────
print("\n🔄 Chargement des données d'inférence...")
LIST_860  = _load_items(
    PROC_DIR/'segmentation'/'test'/'images',
    PROC_DIR/'segmentation'/'test'/'masks',
    'seg_860')
LIST_1000 = _load_items(
    PROC_DIR/'segmentation_new'/'test'/'images',
    PROC_DIR/'segmentation_new'/'test'/'masks',
    'seg_new_1000')

_n_blk = sum(1 for it in LIST_1000 if it['gt_is_black'])
print(f"  SEG_860  (sans no_tumor)  : {len(LIST_860)}")
print(f"  SEG_1000 (avec no_tumor)  : {len(LIST_1000)} "
      f"(dont {_n_blk} no_tumor — exclus du Dice)")

# ── Fonctions d'inférence ──────────────────────────────────────────────────────
def _to_tensor(arr):
    """numpy(H,W) → tensor(1,1,H,W) sur DEVICE sans copie inutile."""
    return torch.from_numpy(arr.copy()).unsqueeze(0).unsqueeze(0).float().to(DEVICE)


def _compute_metrics_per_image(pred_mask, gt_mask):
    """
    Métriques PER IMAGE (numpy 2D).
    Retourne None si GT tout-zéro (no_tumor → exclu).
    """
    if float(np.sum(gt_mask > 0.5)) == 0.:
        return None  # no_tumor → exclu

    p   = (pred_mask > SEG_THRESH).astype(bool).flatten()
    g   = (gt_mask   > 0.5).astype(bool).flatten()
    eps = 1e-8
    TP  = int(np.sum(p & g)); TN = int(np.sum(~p & ~g))
    FP  = int(np.sum(p & ~g)); FN = int(np.sum(~p & g))

    dice = (2*TP+eps)/(2*TP+FP+FN+eps)
    iou  = (TP+eps)/(TP+FP+FN+eps)
    return {
        'dice'         : float(dice), 'iou': float(iou),
        'sensitivity'  : float((TP+eps)/(TP+FN+eps)),
        'specificity'  : float((TN+eps)/(TN+FP+eps)),
        'precision_px' : float((TP+eps)/(TP+FP+eps)),
        'pixel_accuracy': float((TP+TN)/(TP+TN+FP+FN+eps)),
        'bon_emplacement': bool(iou > 0.10 and TP > 0),
        'TP':TP,'TN':TN,'FP':FP,'FN':FN,
    }


def infer_joint_model(item, model):
    """Inférence modèle joint (retourne seg+clf)."""
    img_t = _to_tensor(item['img_arr'])
    with torch.no_grad():
        with _amp():
            seg_l, clf_l = model(img_t)
    probs = torch.softmax(clf_l, 1).squeeze().cpu().numpy()
    seg_p = torch.sigmoid(seg_l).squeeze().cpu().numpy()
    return {
        'pred_class': IDX_TO_CLASS[int(np.argmax(probs))],
        'confidence' : float(probs.max()),
        'all_probs'  : probs.tolist(),
        'seg_proba'  : seg_p,
        'seg_mask'   : (seg_p > SEG_THRESH).astype(np.float32),
    }


def infer_clf_att_model(item, model_clf, model_att):
    """Inférence pipeline CLF → ATT."""
    img_t = _to_tensor(item['img_arr'])
    pred_class = 'N/A'; probs = [0.25]*4

    if model_clf is not None:
        with torch.no_grad():
            with _amp(): out = model_clf(img_t)
        probs = torch.softmax(out, 1).squeeze().cpu().numpy()
        pred_class = IDX_TO_CLASS[int(np.argmax(probs))]

    if pred_class == 'no_tumor':
        seg_mask = np.zeros((224,224), np.float32)
    elif model_att is not None:
        with torch.no_grad():
            with _amp(): seg = model_att(img_t)
        seg_p    = torch.sigmoid(seg).squeeze().cpu().numpy()
        seg_mask = (seg_p > SEG_THRESH).astype(np.float32)
    else:
        seg_mask = np.zeros((224,224), np.float32)

    return {
        'pred_class': pred_class, 'confidence': float(max(probs)),
        'all_probs' : list(probs), 'seg_mask': seg_mask,
    }


def run_inference(image_list, model, mode='joint', model_clf=None, model_att=None,
                   desc=''):
    """
    Inférence sur une liste d'images.
    mode='joint' : utilise model (UNetWithClassifier ou SwinHAFNet)
    mode='clf_att' : utilise model_clf + model_att
    """
    results = []; _t0 = time.time()

    for item in tqdm(image_list, desc=f"  {desc}"):
        if mode == 'joint':
            ir = infer_joint_model(item, model)
        else:
            ir = infer_clf_att_model(item, model_clf, model_att)

        gt = item.get('gt_mask')
        sm = None
        if gt is not None and ir.get('seg_mask') is not None:
            sm = _compute_metrics_per_image(ir['seg_mask'], gt)

        results.append({
            'img_name'    : item['img_name'],
            'true_class'  : item.get('true_class'),
            'label_int'   : item.get('label_int', 0),
            'pred_class'  : ir.get('pred_class','N/A'),
            'confidence'  : ir.get('confidence', 0.),
            'all_probs'   : ir.get('all_probs',  [0.25]*4),
            'correct'     : (ir.get('pred_class')==item.get('true_class')
                             if item.get('true_class') and ir.get('pred_class')!='N/A'
                             else None),
            'has_mask'    : item.get('has_mask', False),
            'gt_is_black' : item.get('gt_is_black', False),
            'seg_metrics' : sm,
            'seg_mask'    : ir.get('seg_mask'),
            'gt_mask'     : gt,
            'img_arr'     : item['img_arr'],
            'dice'        : sm['dice']        if sm else 0.,
            'iou'         : sm['iou']         if sm else 0.,
            'source'      : item.get('source',''),
        })

    elapsed = time.time() - _t0
    print(f"  ⏱️  {len(results)} images | {elapsed:.1f}s "
          f"({elapsed/max(len(results),1)*1000:.1f} ms/img)")
    return results


def build_report(results, name, out_dir):
    """
    Rapport complet avec classification + segmentation + CSV.
    ✅ CORRECTION FileNotFoundError : mkdir avant toute sauvegarde.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)  # ← FIX FileNotFoundError

    print(f"\n{'='*65}")
    print(f"📊 {name} — {len(results)} images")
    print(f"{'='*65}")

    # Classification
    clf_p = [(r['true_class'],r['pred_class']) for r in results
              if r.get('true_class') and r['pred_class']!='N/A']
    y_t  = [x[0] for x in clf_p]; y_p = [x[1] for x in clf_p]
    clf_summ = {}

    if y_t:
        acc  = sum(t==p for t,p in zip(y_t,y_p))/max(len(y_t),1)
        y_ti = [CLASS_TO_IDX.get(t,0) for t in y_t]
        y_pi = [CLASS_TO_IDX.get(p,0) for p in y_p]
        bacc = balanced_accuracy_score(y_ti, y_pi)
        f1_w = f1_score(y_t, y_p, labels=CLASS_NAMES, average='weighted', zero_division=0)

        print(f"\n  ── Classification ({len(y_t)} images) ──")
        print(f"  Accuracy : {acc:.4f} | Balanced Acc : {bacc:.4f} | F1 W : {f1_w:.4f}")
        print(classification_report(y_t, y_p, labels=CLASS_NAMES,
                                     target_names=CLASS_NAMES, zero_division=0))

        _cm = confusion_matrix(y_ti, y_pi, labels=list(range(N_CLASSES)))
        plot_confusion_matrix(_cm, CLASS_NAMES,
                               f'{name}\nAcc={acc:.4f}|BalAcc={bacc:.4f}',
                               out_dir/f'confusion_{name}.png')
        clf_summ = {'accuracy':acc,'balanced_accuracy':bacc,'f1_weighted':f1_w}

    # Segmentation — exclure no_tumor (gt_is_black=True)
    seg_r = [r for r in results
             if not r.get('gt_is_black',False) and r.get('seg_metrics') is not None]
    seg_summ = {}

    if seg_r:
        dices = [r['seg_metrics']['dice']          for r in seg_r]
        ious  = [r['seg_metrics']['iou']           for r in seg_r]
        senss = [r['seg_metrics']['sensitivity']   for r in seg_r]
        specs = [r['seg_metrics']['specificity']   for r in seg_r]
        precs = [r['seg_metrics']['precision_px']  for r in seg_r]
        pxacc = [r['seg_metrics']['pixel_accuracy']for r in seg_r]
        bons  = [r['seg_metrics']['bon_emplacement']for r in seg_r]

        # Méthode agrégée (comparable à l'entraînement)
        TP_t=FP_t=FN_t=TN_t=0
        for r in seg_r:
            sm=r['seg_metrics']
            TP_t+=sm['TP'];FP_t+=sm['FP'];FN_t+=sm['FN'];TN_t+=sm['TN']
        eps=1e-8
        dice_agg=(2*TP_t+eps)/(2*TP_t+FP_t+FN_t+eps)
        iou_agg =(TP_t+eps)/(TP_t+FP_t+FN_t+eps)

        print(f"\n  ── Segmentation ({len(seg_r)} images avec tumeur) ──")
        print(f"\n  [MÉTHODE PAR IMAGE — inference standard]")
        print(f"  Dice moyen : {np.mean(dices):.4f} ± {np.std(dices):.4f}")
        print(f"  IoU moyen  : {np.mean(ious):.4f}  ± {np.std(ious):.4f}")
        print(f"  Sensitivity: {np.mean(senss):.4f} | Specificity: {np.mean(specs):.4f}")
        print(f"  Precision  : {np.mean(precs):.4f} | Pixel Acc  : {np.mean(pxacc):.4f}")
        print(f"  Bon Empl   : {sum(bons)}/{len(bons)} ({sum(bons)/len(bons)*100:.1f}%)")
        print(f"\n  [MÉTHODE AGRÉGÉE — comparable aux courbes entraînement]")
        print(f"  Dice agrégé : {dice_agg:.4f}  ← comparable à dice test courbes")
        print(f"  IoU agrégé  : {iou_agg:.4f}")
        print(f"  ⚠️  Δ Dice = {dice_agg-np.mean(dices):.3f} "
              f"(agrégé > moyen — grandes tumeurs dominent l'agrégé)")

        # Par classe
        by_cls = defaultdict(list)
        for r in seg_r:
            if r.get('true_class'): by_cls[r['true_class']].append(r['seg_metrics'])
        seg_cls_rows = []
        for cls in CLASS_NAMES:
            ms = by_cls.get(cls, [])
            if ms:
                seg_cls_rows.append({'Classe':cls,'N':len(ms),
                    'Dice':f"{np.mean([m['dice'] for m in ms]):.4f}",
                    'IoU' :f"{np.mean([m['iou']  for m in ms]):.4f}",
                    'Sens':f"{np.mean([m['sensitivity'] for m in ms]):.4f}",
                    'Prec':f"{np.mean([m['precision_px'] for m in ms]):.4f}",
                    'BonEmpl':f"{sum(m['bon_emplacement'] for m in ms)}/{len(ms)}"})
            else:
                seg_cls_rows.append({'Classe':cls,'N':0,'Dice':'N/A',
                    'IoU':'N/A','Sens':'N/A','Prec':'N/A','BonEmpl':'N/A'})
        _df_seg_cls = pd.DataFrame(seg_cls_rows)
        print(f"\n  Par classe :\n{_df_seg_cls.to_string(index=False)}")
        _df_seg_cls.to_csv(out_dir/f'seg_par_classe_{name}.csv', index=False)

        seg_summ = {
            'n_images':len(seg_r), 'dice_mean':float(np.mean(dices)),
            'dice_std':float(np.std(dices)), 'iou_mean':float(np.mean(ious)),
            'dice_agrege':float(dice_agg), 'iou_agrege':float(iou_agg),
            'sensitivity':float(np.mean(senss)), 'specificity':float(np.mean(specs)),
            'precision_px':float(np.mean(precs)), 'pixel_acc':float(np.mean(pxacc)),
            'bon_empl_pct':float(sum(bons)/max(len(bons),1)*100),
        }

    # CSV complet
    _rows = []
    for r in results:
        row = {'img_name':r['img_name'],'true_class':r.get('true_class','?'),
               'pred_class':r.get('pred_class','N/A'),'confidence':round(r.get('confidence',0.),4),
               'correct':str(r.get('correct')),'gt_is_black':str(r.get('gt_is_black',False))}
        sm = r.get('seg_metrics') or {}
        for mk in ['dice','iou','sensitivity','specificity','precision_px','pixel_accuracy']:
            row[mk] = round(float(sm.get(mk,float('nan'))),4) if mk in sm else 'N/A'
        _rows.append(row)
    pd.DataFrame(_rows).to_csv(
        out_dir/f'resultats_{name}.csv', index=False, encoding='utf-8-sig')
    print(f"\n  💾 CSV → resultats_{name}.csv ({len(_rows)} lignes)")
    return {'clf':clf_summ,'seg':seg_summ}


# ── Exécution inférence Joint_Total ───────────────────────────────────────────
model_joint.eval()
print("\n📌 Inférence Joint_Total — 1000 images (segmentation_new, avec no_tumor)")
RES_JNT_1000 = run_inference(LIST_1000, model_joint, mode='joint',
                               desc='Joint 1000')
SUMM_JNT_1000 = build_report(RES_JNT_1000, 'joint_1000', JOINT_INF_DIR/'joint_1000')

print("\n📌 Inférence Joint_Total — 860 images (segmentation, sans no_tumor)")
RES_JNT_860 = run_inference(LIST_860, model_joint, mode='joint',
                              desc='Joint 860')
SUMM_JNT_860 = build_report(RES_JNT_860, 'joint_860', JOINT_INF_DIR/'joint_860')

# ── Inférence CLF + ATT (860 images, depuis result_model) ─────────────────────
print("\n📌 Inférence CLF + ATT — 860 images (segmentation/test)")
RES_CLF_ATT = []
if MODEL_CLF is not None or MODEL_ATT is not None:
    for item in tqdm(LIST_860, desc="  CLF+ATT 860"):
        ir  = infer_clf_att_model(item, MODEL_CLF, MODEL_ATT)
        gt  = item.get('gt_mask')
        sm  = _compute_metrics_per_image(ir['seg_mask'], gt) if gt is not None else None
        RES_CLF_ATT.append({
            'img_name'   : item['img_name'],
            'true_class' : item.get('true_class'),
            'pred_class' : ir['pred_class'],
            'confidence' : ir['confidence'],
            'correct'    : (ir['pred_class']==item.get('true_class')
                            if item.get('true_class') and ir['pred_class']!='N/A'
                            else None),
            'has_mask'   : item.get('has_mask',False),
            'gt_is_black': item.get('gt_is_black',False),
            'seg_metrics': sm,
            'seg_mask'   : ir['seg_mask'],
            'gt_mask'    : gt,
            'img_arr'    : item['img_arr'],
            'dice'       : sm['dice'] if sm else 0.,
            'iou'        : sm['iou']  if sm else 0.,
        })
    SUMM_CLF_ATT = build_report(RES_CLF_ATT, 'clf_att_860',
                                  JOINT_INF_DIR/'clf_att_860')
else:
    print("  ⚠️  CLF/ATT non disponibles (modèles non chargés)")
    SUMM_CLF_ATT = {}

# ── 9 prédictions annotées Joint_Total ────────────────────────────────────────
print("\n🖼️  Génération des 9 prédictions Joint_Total...")
plot_9_predictions(RES_JNT_860, 'Joint_Total',
                   JOINT_RESULTS/'joint_total_9_predictions.png')

# ── Superposition de TOUTES les images (860 et 1000) ─────────────────────────
print("\n🖼️  Superposition TOUTES images Joint_Total (860 seg)...")
plot_all_superpositions(RES_JNT_860, 'Joint_860',
                         JOINT_INF_DIR/'joint_860'/'superpositions', batch_size=50)

print("\n🖼️  Superposition TOUTES images Joint_Total (1000 seg_new)...")
plot_all_superpositions(RES_JNT_1000, 'Joint_1000',
                         JOINT_INF_DIR/'joint_1000'/'superpositions', batch_size=50)

if RES_CLF_ATT:
    print("\n🖼️  Superposition TOUTES images CLF+ATT (860 seg)...")
    plot_all_superpositions(RES_CLF_ATT, 'CLF_ATT_860',
                             JOINT_INF_DIR/'clf_att_860'/'superpositions',
                             batch_size=50)

# ── Tableau comparatif Joint_Total vs CLF+ATT ─────────────────────────────────
print("\n📊 TABLEAU COMPARATIF : Joint_Total vs CLF+ATT")
_comp_rows = []
for _name, _summ in [
    ('Joint_Total_860',  SUMM_JNT_860),
    ('Joint_Total_1000', SUMM_JNT_1000),
    ('CLF_ATT_860',      SUMM_CLF_ATT),
]:
    clf = _summ.get('clf',{}); seg = _summ.get('seg',{})
    _comp_rows.append({
        'Pipeline'    : _name,
        'Accuracy'    : round(clf.get('accuracy',         float('nan')), 4),
        'BalAcc'      : round(clf.get('balanced_accuracy',float('nan')), 4),
        'F1_weighted' : round(clf.get('f1_weighted',      float('nan')), 4),
        'Dice_img_moy': round(seg.get('dice_mean',        float('nan')), 4),
        'Dice_agrege' : round(seg.get('dice_agrege',      float('nan')), 4),
        'IoU_img_moy' : round(seg.get('iou_mean',         float('nan')), 4),
        'Sensitivity' : round(seg.get('sensitivity',      float('nan')), 4),
        'BonEmpl_%'   : round(seg.get('bon_empl_pct',     float('nan')), 2),
    })

df_comp_jnt = pd.DataFrame(_comp_rows)
print(df_comp_jnt.to_string(index=False))
df_comp_jnt.to_csv(JOINT_RESULTS/'comparaison_joint_vs_clf_att.csv', index=False)
print(f"\n  💾 → comparaison_joint_vs_clf_att.csv")

print("✅ Bloc 10 terminé")

# ============================================================================
# BLOC 11 : ARCHIVAGE → resultat_joint.zip
# ============================================================================
print("\n" + "="*70)
print("🗜️  BLOC 11 : ARCHIVAGE → resultat_joint.zip")
print("="*70)

# Structure du ZIP résultat_joint :
# processed_donnees/ (données prétraitées)
# models/ (joint_total.pth + pths de result_model)
# results/joint_total/ (courbes, métriques, matrices)
# inference/ (résultats des 3 expériences)

_jnt_archive_dir = KAGGLE_WORKING / 'archive_joint'
_jnt_archive_dir.mkdir(exist_ok=True)

# Copier processed_donnees
print("  Copie processed_donnees...")
shutil.copytree(str(PROC_DIR),
                str(_jnt_archive_dir / 'processed_donnees'),
                dirs_exist_ok=True)

# Copier models
_mdst = _jnt_archive_dir / 'models'
_mdst.mkdir(exist_ok=True)
for pth in MODELS_DIR.glob('*.pth'):
    shutil.copy2(pth, _mdst/pth.name)
print(f"  Modèles copiés : {[p.name for p in _mdst.glob('*.pth')]}")

# Copier résultats
shutil.copytree(str(JOINT_RESULTS),
                str(_jnt_archive_dir / 'resultats'),
                dirs_exist_ok=True)

# Créer le ZIP
_zip_jnt = KAGGLE_WORKING / 'resultat_joint'
print(f"  Création resultat_joint.zip ...")
shutil.make_archive(str(_zip_jnt), 'zip', str(_jnt_archive_dir))
_jnt_size = _zip_jnt.with_suffix('.zip').stat().st_size / 1e6
print(f"  ✅ resultat_joint.zip ({_jnt_size:.1f} Mo)")

print("✅ Bloc 11 terminé")

# ============================================================================
# BLOC 12 : ENTRAÎNEMENT SWINHAFNET (5000 images avec no_tumor)
# ============================================================================
print("\n" + "="*80)
print("🎓 BLOC 12 : ENTRAÎNEMENT SWINHAFNET")
print("="*80)
"""
SwinHAFNet : Swin Transformer (backbone Swin-Tiny ImageNet) + décodeur UNet
Entraîné sur segmentation_new (5000 images avec no_tumor).
Résultats stockés séparément dans resultat_swin.zip.
Checkpoint toutes les 10 époques pour résistance à l'arrêt GPU.
"""

# Dossiers Swin
SWIN_RESULTS = RESULTS_DIR / 'swin'
SWIN_PLOTS   = SWIN_RESULTS / 'plots'
SWIN_INF_DIR = SWIN_RESULTS / 'inference'
for _d in [SWIN_RESULTS, SWIN_PLOTS, SWIN_INF_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# ── Datasets (réutiliser ceux du joint) ──────────────────────────────────────
print("\n📦 Chargement datasets SwinHAFNet...")
full_sw_ds = SegmentationNewDataset(str(PROC_DIR), 'train',
                                     get_seg_transforms('train'))
test_sw_ds = SegmentationNewDataset(str(PROC_DIR), 'test',
                                     get_seg_transforms('test'))

_ts_sw = int(0.8*len(full_sw_ds)); _vs_sw = len(full_sw_ds)-_ts_sw
train_sw_sub, val_sw_sub = random_split(
    full_sw_ds, [_ts_sw, _vs_sw],
    generator=torch.Generator().manual_seed(42))

_bs_sw = HYPERPARAMS['BATCH_SWIN']
train_sw_loader = DataLoader(train_sw_sub, batch_size=_bs_sw,
                              shuffle=True, **_dl_kwargs(True))
val_sw_loader   = DataLoader(val_sw_sub,   batch_size=_bs_sw,
                              shuffle=False, **_dl_kwargs(False))
test_sw_loader  = DataLoader(test_sw_ds,   batch_size=_bs_sw,
                              shuffle=False, **_dl_kwargs(False))

print(f"  Train:{len(train_sw_sub)} | Val:{len(val_sw_sub)} | Test:{len(test_sw_ds)}")

# ── Modèle SwinHAFNet ─────────────────────────────────────────────────────────
print("\n🧠 Initialisation SwinHAFNet...")
_sw_pth      = MODELS_DIR / 'swin_hafnet_best.pth'
_sw_ckpt_dir = MODELS_DIR / 'checkpoints_swin'
_sw_ckpt_dir.mkdir(parents=True, exist_ok=True)

# Reprendre depuis checkpoint si disponible
_sw_start_epoch = 1
model_swin = None
for _ckpt in sorted(_sw_ckpt_dir.glob('checkpoint_ep*.pth'), reverse=True):
    try:
        _m_tmp = SwinHAFNet(n_clf_classes=4, pretrained=False).to(DEVICE)
        _m_tmp.load_state_dict(torch.load(str(_ckpt), map_location=DEVICE))
        model_swin = _m_tmp
        _ep_str = _ckpt.stem.replace('checkpoint_ep','')
        _sw_start_epoch = int(_ep_str) + 1
        print(f"  ♻️  Reprise depuis {_ckpt.name} (epoch {_sw_start_epoch})")
        break
    except Exception:
        continue

if model_swin is None:
    try:
        model_swin = SwinHAFNet(n_clf_classes=4, pretrained=True).to(DEVICE)
        print("  ✅ Backbone Swin-Tiny pré-entraîné ImageNet")
    except Exception as e:
        print(f"  ⚠️  Sans pré-entraînement ({e})")
        model_swin = SwinHAFNet(n_clf_classes=4, pretrained=False).to(DEVICE)

_, total_params_sw = get_model_summary_table(
    model_swin, (1,1,224,224), "SwinHAFNet")

optimizer_sw  = optim.AdamW(model_swin.parameters(),
                              lr=HYPERPARAMS['LR_SWIN'],
                              weight_decay=HYPERPARAMS['WEIGHT_DECAY'])
scheduler_sw  = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_sw, mode='max', patience=5, factor=0.5, min_lr=1e-6)
crit_seg_sw   = CombinedLoss(pos_weight=pos_weight)
crit_clf_sw   = nn.CrossEntropyLoss(weight=class_weights)
scaler_sw     = GradScaler() if USE_AMP else None
early_stop_sw = EarlyStopping(patience=HYPERPARAMS['EARLY_STOP_PATIENCE'],
                                mode='max', save_path=str(_sw_pth))

history_sw = {f'{s}_{m}': []
              for s in ['train','val','test']
              for m in ['loss','acc','f1','iou','dice','pixel_acc']}

t_train_sw=t_val_sw=t_test_sw=0.
best_sw_preds=best_sw_labels=[]
best_sw_seg={}; epochs_done_sw=0

print(f"  Params : {total_params_sw:,} ({total_params_sw/1e6:.2f}M)")
print(f"  LR     : {HYPERPARAMS['LR_SWIN']}")
print(f"  Epochs : {HYPERPARAMS['EPOCHS']} max | patience={HYPERPARAMS['EARLY_STOP_PATIENCE']}")
print("\n🏋️  Entraînement SwinHAFNet...")

for epoch in range(_sw_start_epoch, HYPERPARAMS['EPOCHS']+1):

    # ── TRAIN ──────────────────────────────────────────────────────────────────
    model_swin.train()
    tr_loss=tr_correct=tr_total=0
    _tr_sp,_tr_sg,_tr_cp,_tr_cl=[],[],[],[]
    _t0=time.time()

    for imgs,masks,lbls in tqdm(train_sw_loader,
                                 desc=f"  Ep {epoch:3d} Train", leave=False):
        imgs,masks,lbls=imgs.to(DEVICE),masks.to(DEVICE),lbls.to(DEVICE)
        optimizer_sw.zero_grad(set_to_none=True)
        if USE_AMP:
            with autocast():
                seg_out,clf_out=model_swin(imgs)
                loss=(HYPERPARAMS['SEG_LOSS_W']*crit_seg_sw(seg_out,masks)+
                      HYPERPARAMS['CLF_LOSS_W']*crit_clf_sw(clf_out,lbls))
            scaler_sw.scale(loss).backward()
            scaler_sw.unscale_(optimizer_sw)
            nn.utils.clip_grad_norm_(model_swin.parameters(),1.0)
            scaler_sw.step(optimizer_sw); scaler_sw.update()
        else:
            seg_out,clf_out=model_swin(imgs)
            loss=(HYPERPARAMS['SEG_LOSS_W']*crit_seg_sw(seg_out,masks)+
                  HYPERPARAMS['CLF_LOSS_W']*crit_clf_sw(clf_out,lbls))
            loss.backward()
            nn.utils.clip_grad_norm_(model_swin.parameters(),1.0)
            optimizer_sw.step()

        tr_loss+=loss.item(); _,pred=torch.max(clf_out,1)
        tr_correct+=(pred==lbls).sum().item(); tr_total+=lbls.size(0)
        _tr_cp.extend(pred.cpu().numpy().tolist())
        _tr_cl.extend(lbls.cpu().numpy().tolist())
        with torch.no_grad():
            _tr_sp.append(torch.sigmoid(seg_out.detach()).cpu())
            _tr_sg.append(masks.cpu())

    t_train_sw+=time.time()-_t0
    tr_loss/=len(train_sw_loader); tr_acc=tr_correct/max(tr_total,1)
    tr_f1=f1_score(_tr_cl,_tr_cp,labels=CLF_LABELS,average='weighted',zero_division=0)
    tr_seg=calculate_seg_metrics_full(_tr_sp,_tr_sg,exclude_black_masks=True)

    # ── VALIDATION ─────────────────────────────────────────────────────────────
    model_swin.eval()
    v_loss=v_correct=v_total=0
    _v_sp,_v_sg,_v_cp,_v_cl=[],[],[],[]
    _t0=time.time()
    with torch.no_grad():
        for imgs,masks,lbls in val_sw_loader:
            imgs,masks,lbls=imgs.to(DEVICE),masks.to(DEVICE),lbls.to(DEVICE)
            with _amp():
                seg_out,clf_out=model_swin(imgs)
                loss=(HYPERPARAMS['SEG_LOSS_W']*crit_seg_sw(seg_out,masks)+
                      HYPERPARAMS['CLF_LOSS_W']*crit_clf_sw(clf_out,lbls))
            v_loss+=loss.item(); _,pred=torch.max(clf_out,1)
            v_correct+=(pred==lbls).sum().item(); v_total+=lbls.size(0)
            _v_cp.extend(pred.cpu().numpy().tolist())
            _v_cl.extend(lbls.cpu().numpy().tolist())
            _v_sp.append(torch.sigmoid(seg_out).cpu()); _v_sg.append(masks.cpu())

    t_val_sw+=time.time()-_t0
    v_loss/=len(val_sw_loader); v_acc=v_correct/max(v_total,1)
    v_f1=f1_score(_v_cl,_v_cp,labels=CLF_LABELS,average='weighted',zero_division=0)
    v_seg=calculate_seg_metrics_full(_v_sp,_v_sg,exclude_black_masks=True)
    _cat_sp=torch.cat(_v_sp[:10]); _cat_sg=torch.cat(_v_sg[:10])
    v_px=float(np.mean([calculate_seg_metrics(
        _cat_sp[i:i+1],_cat_sg[i:i+1])[2]
        for i in range(min(50,len(_cat_sp)))]))

    # ── TEST (monitoring) ──────────────────────────────────────────────────────
    te_loss=te_correct=te_total=0
    _te_sp,_te_sg,_te_cp,_te_cl=[],[],[],[]
    _t0=time.time()
    with torch.no_grad():
        for imgs,masks,lbls in test_sw_loader:
            imgs,masks,lbls=imgs.to(DEVICE),masks.to(DEVICE),lbls.to(DEVICE)
            with _amp():
                seg_out,clf_out=model_swin(imgs)
                loss=(HYPERPARAMS['SEG_LOSS_W']*crit_seg_sw(seg_out,masks)+
                      HYPERPARAMS['CLF_LOSS_W']*crit_clf_sw(clf_out,lbls))
            te_loss+=loss.item(); _,pred=torch.max(clf_out,1)
            te_correct+=(pred==lbls).sum().item(); te_total+=lbls.size(0)
            _te_cp.extend(pred.cpu().numpy().tolist())
            _te_cl.extend(lbls.cpu().numpy().tolist())
            _te_sp.append(torch.sigmoid(seg_out).cpu()); _te_sg.append(masks.cpu())

    t_test_sw+=time.time()-_t0
    te_loss/=max(len(test_sw_loader),1); te_acc=te_correct/max(te_total,1)
    te_f1=f1_score(_te_cl,_te_cp,labels=CLF_LABELS,average='weighted',zero_division=0)
    te_seg=calculate_seg_metrics_full(_te_sp,_te_sg,exclude_black_masks=True)

    # Historique
    for split,(l_,a_,f_,seg_,px_) in [
        ('train',(tr_loss,tr_acc,tr_f1,tr_seg,0.)),
        ('val'  ,(v_loss, v_acc, v_f1, v_seg, v_px)),
        ('test' ,(te_loss,te_acc,te_f1,te_seg,0.)),
    ]:
        history_sw[f'{split}_loss'     ].append(l_)
        history_sw[f'{split}_acc'      ].append(a_)
        history_sw[f'{split}_f1'       ].append(f_)
        history_sw[f'{split}_iou'      ].append(seg_.get('IoU', 0.))
        history_sw[f'{split}_dice'     ].append(seg_.get('Dice',0.))
        history_sw[f'{split}_pixel_acc'].append(px_ if split=='val' else 0.)

    combined=0.5*v_seg.get('IoU',0.)+0.5*v_acc
    scheduler_sw.step(combined)

    print(f"\n  Ep {epoch:3d} | Train Loss:{tr_loss:.4f} Acc:{tr_acc:.4f} "
          f"F1:{tr_f1:.4f} IoU:{tr_seg.get('IoU',0):.4f} Dice:{tr_seg.get('Dice',0):.4f}")
    print(f"         | Val   Loss:{v_loss:.4f}  Acc:{v_acc:.4f}  "
          f"F1:{v_f1:.4f}  IoU:{v_seg.get('IoU',0):.4f}  Dice:{v_seg.get('Dice',0):.4f} "
          f"PxAcc:{v_px:.4f}")
    print(f"         | Test  Loss:{te_loss:.4f} Acc:{te_acc:.4f} "
          f"F1:{te_f1:.4f} IoU:{te_seg.get('IoU',0):.4f} Dice:{te_seg.get('Dice',0):.4f}")
    print(f"         | Combined:{combined:.4f} LR:{optimizer_sw.param_groups[0]['lr']:.2e}")

    if combined > getattr(early_stop_sw,'_best_combined',0.):
        early_stop_sw._best_combined=combined
        best_sw_preds=list(_te_cp); best_sw_labels=list(_te_cl); best_sw_seg=te_seg

    # Checkpoint toutes les 10 époques
    if epoch % 10 == 0:
        _ckpt_p = _sw_ckpt_dir / f'checkpoint_ep{epoch:03d}.pth'
        torch.save(model_swin.state_dict(), _ckpt_p)
        pd.DataFrame(history_sw).to_csv(
            _sw_ckpt_dir/f'history_ep{epoch:03d}.csv',index=False)
        print(f"  💾 Checkpoint Swin → {_ckpt_p.name}")

    if early_stop_sw(combined,model_swin):
        print(f"\n  ⚠️  Early stopping Swin epoch {epoch}")
        epochs_done_sw=epoch; break
    epochs_done_sw=epoch
    if DEVICE.type=='cuda' and epoch%5==0: torch.cuda.empty_cache()

print(f"\n✅ SwinHAFNet terminé — {epochs_done_sw} époques | Best:{early_stop_sw.best:.4f}")

if _sw_pth.exists():
    model_swin.load_state_dict(torch.load(str(_sw_pth),map_location=DEVICE))
model_swin.eval()

# ── Courbes SwinHAFNet ─────────────────────────────────────────────────────────
print("\n📊 Génération courbes SwinHAFNet...")
plot_training_curves(history_sw,'SwinHAFNet (5000 img)',
                     SWIN_PLOTS/'swin_training_curves.png')
for _mk,_yl in [('loss','Loss'),('iou','IoU'),('dice','Dice'),
                 ('f1','F1-Score'),('acc','Accuracy')]:
    plot_metric_curve(history_sw,'SwinHAFNet',_mk,
                       SWIN_PLOTS/f'swin_{_mk}_train_val_test.png',
                       ylabel=_yl,title=f'SwinHAFNet — {_yl} (Train/Val/Test)')

pd.DataFrame(history_sw).to_csv(SWIN_RESULTS/'swin_history.csv',index=False)

# ── Évaluation finale SwinHAFNet ──────────────────────────────────────────────
print("\n📊 ÉVALUATION FINALE SwinHAFNet sur test (1000 images, avec no_tumor) :")
model_swin.eval()
_sw_fp,_sw_fl,_sw_sp,_sw_sg=[],[],[],[]
te_dice_sw=te_iou_sw=te_px_sw=te_n_sw=0.
with torch.no_grad():
    for imgs,masks,lbls in tqdm(test_sw_loader,desc="  Final Swin"):
        imgs,masks=imgs.to(DEVICE),masks.to(DEVICE)
        with _amp():
            seg_out,clf_out=model_swin(imgs)
        _,pred=torch.max(clf_out,1)
        _sw_fp.extend(pred.cpu().numpy().tolist())
        _sw_fl.extend(lbls.numpy().tolist())
        d,i,p=calculate_seg_metrics(torch.sigmoid(seg_out),masks)
        te_dice_sw+=d; te_iou_sw+=i; te_px_sw+=p; te_n_sw+=1
        _sw_sp.append(torch.sigmoid(seg_out).cpu()); _sw_sg.append(masks.cpu())

te_dice_sw/=max(te_n_sw,1); te_iou_sw/=max(te_n_sw,1); te_px_sw/=max(te_n_sw,1)
_sw_full=calculate_seg_metrics_full(_sw_sp,_sw_sg,exclude_black_masks=True)

print(f"\n  === MÉTRIQUES FINALES SwinHAFNet (test 1000 images) ===")
print(f"  Clf Accuracy : {(np.array(_sw_fp)==np.array(_sw_fl)).mean():.4f}")
print(f"  Seg IoU      : {_sw_full.get('IoU', te_iou_sw):.4f}")
print(f"  Seg Dice     : {_sw_full.get('Dice',te_dice_sw):.4f}")
print(f"  Sensitivity  : {_sw_full.get('Sensitivity',float('nan')):.4f}")
print(f"  Specificity  : {_sw_full.get('Specificity',float('nan')):.4f}")
print(f"  Precision    : {_sw_full.get('Precision',  float('nan')):.4f}")
print(f"  Pixel_Acc    : {_sw_full.get('Pixel_Acc',  te_px_sw):.4f}")

print("\n📋 Classification Report — SwinHAFNet (avec no_tumor) :")
_sw_fl_str=[CLASS_NAMES[l] for l in _sw_fl]; _sw_fp_str=[CLASS_NAMES[p] for p in _sw_fp]
print(classification_report(_sw_fl_str,_sw_fp_str,
                              labels=CLASS_NAMES,target_names=CLASS_NAMES,zero_division=0))

_cm_sw=confusion_matrix(_sw_fl,_sw_fp,labels=list(range(N_CLASSES)))
plot_confusion_matrix(_cm_sw,CLASS_NAMES,'SwinHAFNet — Test 1000 (avec no_tumor)',
                       SWIN_PLOTS/'swin_confusion_matrix.png')

# Stocker globaux
results_all['swin_clf']=classification_report(_sw_fl_str,_sw_fp_str,
    labels=CLASS_NAMES,target_names=CLASS_NAMES,zero_division=0,output_dict=True)
results_all['swin_seg']=best_sw_seg or _sw_full
all_histories['SwinHAFNet']=history_sw
complexity_info['SwinHAFNet']={
    'Params (M)':round(total_params_sw/1e6,3),
    'Train_Time(s)':round(t_train_sw,1),
    'Val_Time(s)':round(t_val_sw,1),
    'Test_Time(s)':round(t_test_sw,1),
}

print("✅ Bloc 12 terminé")

# ============================================================================
# BLOC 13 : INFÉRENCE SWINHAFNET + VISUALISATIONS
# ============================================================================
print("\n" + "="*80)
print("🔬 BLOC 13 : INFÉRENCE SWINHAFNET")
print("="*80)

model_swin.eval()

# Inférence 1000 images
print("\n📌 Inférence Swin — 1000 images (segmentation_new, avec no_tumor)")
RES_SW_1000 = run_inference(LIST_1000, model_swin, mode='joint', desc='Swin 1000')
SUMM_SW_1000 = build_report(RES_SW_1000,'swin_1000', SWIN_INF_DIR/'swin_1000')

# Inférence 860 images
print("\n📌 Inférence Swin — 860 images (segmentation, sans no_tumor)")
RES_SW_860 = run_inference(LIST_860, model_swin, mode='joint', desc='Swin 860')
SUMM_SW_860 = build_report(RES_SW_860,'swin_860', SWIN_INF_DIR/'swin_860')

# 9 prédictions annotées SwinHAFNet
print("\n🖼️  9 prédictions SwinHAFNet...")
plot_9_predictions(RES_SW_860, 'SwinHAFNet',
                   SWIN_RESULTS/'swin_9_predictions.png')

# Superposition toutes images
print("\n🖼️  Superposition TOUTES images Swin (860)...")
plot_all_superpositions(RES_SW_860, 'Swin_860',
                         SWIN_INF_DIR/'swin_860'/'superpositions', batch_size=50)

print("\n🖼️  Superposition TOUTES images Swin (1000)...")
plot_all_superpositions(RES_SW_1000, 'Swin_1000',
                         SWIN_INF_DIR/'swin_1000'/'superpositions', batch_size=50)

print("✅ Bloc 13 terminé")

# ============================================================================
# BLOC 14 : TABLEAU COMPARATIF FINAL (3 pipelines)
# ============================================================================
print("\n" + "="*80)
print("📊 BLOC 14 : TABLEAU COMPARATIF FINAL — 5 EXPÉRIENCES")
print("="*80)

_all_rows = []
for _pname, _summ in [
    ('CLF+ATT (860)',        SUMM_CLF_ATT),
    ('Joint_Total (860)',    SUMM_JNT_860),
    ('Joint_Total (1000+T)', SUMM_JNT_1000),
    ('Swin (860)',           SUMM_SW_860),
    ('Swin (1000+T)',        SUMM_SW_1000),
]:
    clf=_summ.get('clf',{}); seg=_summ.get('seg',{})
    _all_rows.append({
        'Pipeline'    : _pname,
        'Accuracy'    : round(clf.get('accuracy',         float('nan')),4),
        'BalAcc'      : round(clf.get('balanced_accuracy',float('nan')),4),
        'F1_weighted' : round(clf.get('f1_weighted',      float('nan')),4),
        'Dice_img_moy': round(seg.get('dice_mean',        float('nan')),4),
        'Dice_agrege' : round(seg.get('dice_agrege',      float('nan')),4),
        'IoU_moy'     : round(seg.get('iou_mean',         float('nan')),4),
        'Sensitivity' : round(seg.get('sensitivity',      float('nan')),4),
        'Specificity' : round(seg.get('specificity',      float('nan')),4),
        'BonEmpl_%'   : round(seg.get('bon_empl_pct',     float('nan')),2),
    })

df_final = pd.DataFrame(_all_rows)
print(df_final.to_string(index=False))
_comp_dir = RESULTS_DIR / 'comparaison'
_comp_dir.mkdir(exist_ok=True)
df_final.to_csv(_comp_dir/'comparaison_5_experiences.csv', index=False)

# Heatmap de comparaison
_hcols = ['Accuracy','Dice_img_moy','Dice_agrege','IoU_moy','Sensitivity','Specificity']
_heat  = df_final[['Pipeline']+_hcols].set_index('Pipeline')
_heat  = _heat.apply(pd.to_numeric, errors='coerce')
if not _heat.empty and not _heat.isna().all().all():
    fig, ax = plt.subplots(figsize=(14, max(4, len(_all_rows)*1.2)+1))
    sns.heatmap(_heat, annot=True, fmt='.4f', cmap='YlOrRd',
                linewidths=0.5, ax=ax, vmin=0.5, vmax=1.0, annot_kws={'size':10})
    ax.set_title('Comparaison 5 Expériences — Toutes Métriques',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(_comp_dir/'heatmap_5_experiences.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"\n  💾 Heatmap → heatmap_5_experiences.png")

# Tableau complexité computationnelle
print("\n⏱️  COMPLEXITÉ COMPUTATIONNELLE :")
df_complexity = pd.DataFrame(complexity_info).T
df_complexity.index.name = 'Modèle'
print(df_complexity.to_string())
df_complexity.to_csv(_comp_dir/'complexite_computationnelle.csv')

print("✅ Bloc 14 terminé")

# ============================================================================
# BLOC 15 : ARCHIVAGE → resultat_swin.zip
# ============================================================================
print("\n" + "="*70)
print("🗜️  BLOC 15 : ARCHIVAGE → resultat_swin.zip")
print("="*70)

_sw_archive_dir = KAGGLE_WORKING / 'archive_swin'
_sw_archive_dir.mkdir(exist_ok=True)

# Modèle SwinHAFNet
_sw_mdir = _sw_archive_dir / 'model'
_sw_mdir.mkdir(exist_ok=True)
if _sw_pth.exists():
    shutil.copy2(_sw_pth, _sw_mdir / _sw_pth.name)
# Checkpoints
for _ck in _sw_ckpt_dir.glob('*.pth'):
    shutil.copy2(_ck, _sw_mdir / _ck.name)

# Résultats
shutil.copytree(str(SWIN_RESULTS), str(_sw_archive_dir/'resultats'), dirs_exist_ok=True)

# Tableau comparatif final
shutil.copytree(str(_comp_dir), str(_sw_archive_dir/'comparaison'), dirs_exist_ok=True)

# ZIP
_zip_sw = KAGGLE_WORKING / 'resultat_swin'
print(f"  Création resultat_swin.zip ...")
shutil.make_archive(str(_zip_sw), 'zip', str(_sw_archive_dir))
_sw_size = _zip_sw.with_suffix('.zip').stat().st_size / 1e6
print(f"  ✅ resultat_swin.zip ({_sw_size:.1f} Mo)")

print("✅ Bloc 15 terminé")

# ============================================================================
# RÉSUMÉ FINAL
# ============================================================================
print("\n" + "="*80)
print("✅ PIPELINE COMPLET TERMINÉ")
print("="*80)

print(f"""
  📦 SORTIES :
     /kaggle/working/resultat_joint.zip  ({(KAGGLE_WORKING/'resultat_joint.zip').stat().st_size/1e6:.1f} Mo)
       ├── processed_donnees/   (données prétraitées + segmentation_new)
       ├── models/              (joint_total.pth + pths result_model)
       ├── resultats/joint_total/ (courbes + métriques + confusion)
       └── inference/           (3 expériences : CLF+ATT, Joint×2)

     /kaggle/working/resultat_swin.zip   ({(KAGGLE_WORKING/'resultat_swin.zip').stat().st_size/1e6:.1f} Mo)
       ├── model/               (swin_hafnet_best.pth + checkpoints)
       ├── resultats/swin/      (courbes + métriques + confusion)
       ├── inference/           (2 expériences : Swin×2)
       └── comparaison/         (5 expériences tableau + heatmap)

  📊 COURBES GÉNÉRÉES (identiques code original) :
     Joint_Total : 5 courbes (loss/iou/dice/f1/acc) + training_curves
     SwinHAFNet  : 5 courbes (loss/iou/dice/f1/acc) + training_curves

  🔍 MÉTRIQUES INFÉRENCE (2 méthodes, explication de la différence) :
     Dice par image  : mean(Dice_i)    → ~0.82 (petites tumeurs font baisser)
     Dice agrégé     : 2ΣTP/(2ΣTP+FP+FN) → ~0.87 (grandes tumeurs dominent)
     Δ typique = 0.05 → NORMAL, même preprocessing, même modèle eval()
""")

# Vérification des fichiers générés
print("\n📁 FICHIERS GÉNÉRÉS :")
for _f in [KAGGLE_WORKING/'resultat_joint.zip', KAGGLE_WORKING/'resultat_swin.zip']:
    if _f.exists():
        print(f"  ✅ {_f.name} ({_f.stat().st_size/1e6:.1f} Mo)")
    else:
        print(f"  ❌ {_f.name} ABSENT")

✅ PyTorch 2.10.0+cu128 | CUDA True
✅ GPU : Tesla T4 (15.6 GB)

✅ result_model trouvé : /kaggle/input/datasets/rimbenameur/result-model
   processed_data : /kaggle/input/datasets/rimbenameur/result-model/processed_data
   models         : /kaggle/input/datasets/rimbenameur/result-model/models
   unet_clf_best.pth         : ✅ 38.2 Mo
   att_unet_best.pth         : ✅ 125.7 Mo
   unet_joint_best.pth       : ✅ 126.3 Mo
   unet_seg_best.pth         : ✅ 124.2 Mo

📦 Copie des modèles .pth vers WORK_DIR/models ...
  Copié : unet_clf_best.pth
  Copié : att_unet_best.pth
  Copié : unet_joint_best.pth
  Copié : unet_seg_best.pth
✅ Bloc 1 terminé

📂 BLOC 2 : VÉRIFICATION ET COPIE DES DONNÉES PRÉTRAITÉES
  Copie processed_data → processed_donnees ...
  Copié : segmentation/ (9586 .npy)
  Copié : test/ (1000 .npy)
  Copié : train/ (5000 .npy)
  ✅ Copie terminée

📊 VÉRIFICATION DES DONNÉES :
  ✅ classification/train           :  5000  (attendu [4900-5100])
  ✅ classification/test            :  1000  (


  Ep   1 | Train Loss:1.3134 Acc:0.3905 F1:0.3816 IoU:0.0791 Dice:0.1466
         | Val   Loss:1.2113 Acc:0.4620  F1:0.4187  IoU:0.1423  Dice:0.2491 PxAcc:0.9510
         | Test  Loss:1.1546 Acc:0.5310 F1:0.4678 IoU:0.1970 Dice:0.3292
         | Combined:0.3021 LR:5.00e-04



  Ep   2 | Train Loss:1.1491 Acc:0.4743 F1:0.4554 IoU:0.1575 Dice:0.2722
         | Val   Loss:1.1840 Acc:0.4340  F1:0.4132  IoU:0.1469  Dice:0.2561 PxAcc:0.9666
         | Test  Loss:1.1169 Acc:0.5040 F1:0.4941 IoU:0.1907 Dice:0.3203
         | Combined:0.2904 LR:5.00e-04



  Ep   3 | Train Loss:1.0718 Acc:0.5122 F1:0.4989 IoU:0.1982 Dice:0.3309
         | Val   Loss:1.0137 Acc:0.5580  F1:0.5350  IoU:0.2232  Dice:0.3649 PxAcc:0.9727
         | Test  Loss:0.9666 Acc:0.5720 F1:0.5160 IoU:0.2742 Dice:0.4304
         | Combined:0.3906 LR:5.00e-04



  Ep   4 | Train Loss:0.9752 Acc:0.5595 F1:0.5473 IoU:0.2618 Dice:0.4149
         | Val   Loss:0.9512 Acc:0.5330  F1:0.5224  IoU:0.2625  Dice:0.4159 PxAcc:0.9783
         | Test  Loss:0.9008 Acc:0.6140 F1:0.6167 IoU:0.3219 Dice:0.4870
         | Combined:0.3978 LR:5.00e-04



  Ep   5 | Train Loss:0.9075 Acc:0.5797 F1:0.5716 IoU:0.3171 Dice:0.4815
         | Val   Loss:0.8421 Acc:0.6390  F1:0.6263  IoU:0.3577  Dice:0.5269 PxAcc:0.9886
         | Test  Loss:0.8360 Acc:0.6550 F1:0.6322 IoU:0.4065 Dice:0.5780
         | Combined:0.4984 LR:5.00e-04


... (207 outputs left out)

  Swin 1000: 100%|██████████| 1000/1000 [00:23<00:00, 42.54it/s]


  ⏱️  1000 images | 23.5s (23.5 ms/img)

📊 swin_1000 — 1000 images

  ── Classification (1000 images) ──
  Accuracy : 0.9930 | Balanced Acc : 0.9937 | F1 W : 0.9930
              precision    recall  f1-score   support

      glioma       1.00      0.99      0.99       254
  meningioma       0.99      0.99      0.99       306
    no_tumor       0.99      1.00      1.00       140
   pituitary       0.99      0.99      0.99       300

    accuracy                           0.99      1000
   macro avg       0.99      0.99      0.99      1000
weighted avg       0.99      0.99      0.99      1000

  💾 Matrice → confusion_swin_1000.png

  ── Segmentation (860 images avec tumeur) ──

  [MÉTHODE PAR IMAGE — inference standard]
  Dice moyen : 0.8594 ± 0.1770
  IoU moyen  : 0.7836  ± 0.1986
  Sensitivity: 0.8949 | Specificity: 0.9981
  Precision  : 0.8592 | Pixel Acc  : 0.9961
  Bon Empl   : 841/860 (97.8%)

  [MÉTHODE AGRÉGÉE — comparable aux courbes entraînement]
  Dice agrégé : 0.8969  ← comp

  Swin 860: 100%|██████████| 860/860 [00:20<00:00, 42.83it/s]


  ⏱️  860 images | 20.1s (23.4 ms/img)

📊 swin_860 — 860 images

  ── Classification (860 images) ──
  Accuracy : 0.9919 | Balanced Acc : 0.9917 | F1 W : 0.9924
              precision    recall  f1-score   support

      glioma       1.00      0.99      0.99       254
  meningioma       0.99      0.99      0.99       306
    no_tumor       0.00      0.00      0.00         0
   pituitary       0.99      0.99      0.99       300

    accuracy                           0.99       860
   macro avg       0.74      0.74      0.74       860
weighted avg       0.99      0.99      0.99       860

  💾 Matrice → confusion_swin_860.png

  ── Segmentation (860 images avec tumeur) ──

  [MÉTHODE PAR IMAGE — inference standard]
  Dice moyen : 0.8594 ± 0.1770
  IoU moyen  : 0.7836  ± 0.1986
  Sensitivity: 0.8949 | Specificity: 0.9981
  Precision  : 0.8592 | Pixel Acc  : 0.9961
  Bon Empl   : 841/860 (97.8%)

  [MÉTHODE AGRÉGÉE — comparable aux courbes entraînement]
  Dice agrégé : 0.8969  ← comparabl